# Project Setup for Colab and Kaggle

This notebook was automatically bundled for cloud execution. Run the cell below to reconstruct the project structure and install dependencies.

In [ ]:
# =========================================================
# CLOUD ENVIRONMENT SETUP (AUTO-GENERATED)
# =========================================================
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print("Running in Cloud Environment")
    
    # Write supporting files
    FILES = {
        'config.py': "from pathlib import Path\nfrom dataclasses import dataclass\nfrom typing import Optional, List, Tuple\n\n@dataclass\nclass Config:\n    device: str = 'cuda'\n    seed: int = 42\n    dataset_path: Path = Path('./dataset/humanml3d-subset')\n    output_path: Path = Path('./generation')\n    checkpoint_dir: Path = Path('./checkpoints')\n    motion_dim: int = 263\n    num_joints: int = 22\n    joint_dim: int = 3\n    max_motion_length: int = 200\n    fps: int = 20\n    feature_dims: tuple[slice, ...] = (slice(0, 4), slice(4, 67), slice(193, 259), slice(259, 263))\n    dataset_name: str = 't2m'\n    unit_length: int = 5\n    text_embedding_dim: int = 512\n    text_projection_dim: int = 64\n    joint_feature_projection_dim: int = 64\n    per_joint_out_dim: int = 64\n    model_dim: int = 256\n    num_encoder_layers: int = 2\n    dropout: float = 0.1\n    bidirectional_gru: bool = False\n    num_flow_layers: int = 4\n    num_heads: int = 4\n    batch_size: int = 200\n    learning_rate: float = 0.0001\n    num_epochs: int = 400\n    weight_decay: float = 1e-05\n    gradient_clip: float = 1.0\n    warmup_steps: int = 1000\n    lr_decay: float = 0.95\n    lr_decay_epoch: int = 10\n    flow_loss_weight: float = 1.0\n    context_loss_weight: float = 0.1\n    num_inference_steps: int = 50\n    guidance_scale: float = 1.0\n    num_workers: int = 0\n    pin_memory: bool = True\n    log_interval: int = 50\n    save_interval: int = 5\n    eval_interval: int = 1\n    num_eval_samples: int = 100\n    eval_batch_size: int = 32\n\n    def __post_init__(self):\n        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)\n        self.output_path.mkdir(parents=True, exist_ok=True)\n        self.dataset_path.mkdir(parents=True, exist_ok=True)\n\n    @property\n    def context_encoder_output_dim(self) -> int:\n        return self.model_dim * (2 if self.bidirectional_gru else 1)\n\n    def to_dict(self) -> dict:\n        return {k: str(v) if isinstance(v, Path) else v for k, v in self.__dict__.items()}",
        'models.py': "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom pathlib import Path\nfrom typing import Optional, List, Tuple, Union, Callable, Any\nfrom config import Config\nfrom utils.motion_utils import features_to_positions, preprocess_sequence, get_dataset_config, IncrementalFeatureExtractor\n\nclass MotionHistoryEncoder(nn.Module):\n\n    def __init__(self, frame_feature_dim: int, text_embedding_dim: int, joint_feature_projection_dim: int, text_projection_dim: int, per_joint_out_dim: int, joint_count: int=22, model_dim: int=256, num_layers: int=1, bidirectional: bool=False) -> None:\n        super().__init__()\n        self.frame_feature_dim = frame_feature_dim\n        self.text_embedding_dim = text_embedding_dim\n        self.joint_feature_projection_dim = joint_feature_projection_dim\n        self.text_projection_dim = text_projection_dim\n        self.per_joint_out_dim = per_joint_out_dim\n        self.model_dim = model_dim\n        self.num_layers = num_layers\n        self.bidirectional = bidirectional\n        self.joint_count = joint_count\n        self.text_projection = nn.Linear(text_embedding_dim, text_projection_dim)\n        joint_input_dim = 3 + 3 + 8 + text_projection_dim\n        self.per_joint_encoder = nn.Sequential(nn.Linear(joint_input_dim, joint_feature_projection_dim * 2), nn.SiLU(), nn.Linear(joint_feature_projection_dim * 2, joint_feature_projection_dim))\n        self.gru = nn.GRU(input_size=joint_feature_projection_dim * joint_count, hidden_size=model_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)\n        num_directions = 2 if bidirectional else 1\n        gru_hidden_dim = num_layers * num_directions * model_dim\n        intermediate_dim = model_dim * 2\n        self.per_joint_head = nn.Sequential(nn.Linear(gru_hidden_dim, intermediate_dim), nn.SiLU(), nn.Linear(intermediate_dim, joint_count * per_joint_out_dim))\n        self.null_history = nn.Parameter(torch.zeros(1, 1, frame_feature_dim))\n        self.null_duration = nn.Parameter(torch.zeros(1, 1))\n        self.null_text_embedding = nn.Parameter(torch.zeros(1, text_embedding_dim))\n\n    def forward(self, text: Optional[torch.Tensor], input_features: Optional[torch.Tensor]=None, total_duration: Optional[torch.Tensor]=None, batch_size: Optional[int]=None) -> torch.Tensor:\n        if input_features is not None:\n            B = input_features.shape[0]\n        elif text is not None:\n            B = text.shape[0]\n        elif total_duration is not None:\n            B = total_duration.shape[0]\n        elif batch_size is not None:\n            B = batch_size\n        else:\n            B = 1\n        if text is None:\n            text_embeddings = self.null_text_embedding.expand(B, -1)\n        elif text.shape[0] != B:\n            if text.shape[0] == 1:\n                text_embeddings = text.expand(B, -1)\n            else:\n                raise ValueError(f'Batch mismatch: text tensor({text.shape[0]}) vs batch({B})')\n        else:\n            text_embeddings = text\n        text_projected: torch.Tensor = self.text_projection(text_embeddings)\n        if input_features is None:\n            input_features = self.null_history.expand(B, 1, -1)\n        _, T_hist, _ = input_features.shape\n        text_projected_expanded = text_projected.unsqueeze(1).unsqueeze(2).expand(B, T_hist, self.joint_count, -1)\n        ric_joints = input_features[:, :, 4:67]\n        ric_joints = ric_joints.view(B, T_hist, self.joint_count - 1, 3)\n        root_ric = torch.zeros((B, T_hist, 1, 3), device=ric_joints.device)\n        ric_joints = torch.cat([root_ric, ric_joints], dim=2)\n        ric_vel = input_features[:, :, 193:259]\n        ric_vel = ric_vel.view(B, T_hist, self.joint_count, 3)\n        global_features = torch.cat([input_features[:, :, 0:4], input_features[:, :, 259:263]], dim=-1)\n        global_features = global_features.unsqueeze(-2).expand(B, T_hist, self.joint_count, -1)\n        motion_tokens = torch.cat([ric_joints, ric_vel, global_features, text_projected_expanded], dim=-1)\n        fused_tokens: torch.Tensor = self.per_joint_encoder(motion_tokens)\n        temporal_input = fused_tokens.view(B, T_hist, -1)\n        _, final_hidden_raw = self.gru(temporal_input)\n        final_hidden_vec = final_hidden_raw.transpose(0, 1).reshape(B, -1)\n        output: torch.Tensor = self.per_joint_head(final_hidden_vec)\n        joint_features = output.view(B, self.joint_count, self.per_joint_out_dim)\n        return joint_features\n\n    @property\n    def output_dim(self) -> int:\n        return self.model_dim * (2 if self.bidirectional else 1)\n\nclass KinematicChainEncoder(nn.Module):\n\n    def __init__(self, model_dim: int) -> None:\n        super().__init__()\n        joint_to_chain = [0] * 22\n        joint_to_depth = [0] * 22\n        for d, j in enumerate([0, 2, 5, 8, 11]):\n            joint_to_chain[j], joint_to_depth[j] = (0, d)\n        for d, j in enumerate([1, 4, 7, 10], 1):\n            joint_to_chain[j], joint_to_depth[j] = (1, d)\n        for d, j in enumerate([3, 6, 9, 12, 15], 1):\n            joint_to_chain[j], joint_to_depth[j] = (2, d)\n        for d, j in enumerate([14, 17, 19, 21], 4):\n            joint_to_chain[j], joint_to_depth[j] = (3, d)\n        for d, j in enumerate([13, 16, 18, 20], 4):\n            joint_to_chain[j], joint_to_depth[j] = (4, d)\n        self.register_buffer('joint_to_chain', torch.tensor(joint_to_chain))\n        self.register_buffer('joint_to_depth', torch.tensor(joint_to_depth))\n        self.chain_emb = nn.Embedding(5, model_dim // 2)\n        self.depth_emb = nn.Embedding(8, model_dim // 2)\n\n    def forward(self, joint_ids: torch.Tensor) -> torch.Tensor:\n        chains = self.joint_to_chain[joint_ids]\n        depths = self.joint_to_depth[joint_ids]\n        return torch.cat([self.chain_emb(chains), self.depth_emb(depths)], dim=-1)\n\nclass FlowMatchingPredictor(nn.Module):\n\n    def __init__(self, per_joint_dim: int=32, model_dim: int=64, num_layers: int=2, joint_count: int=22, time_embed_dim: int=64) -> None:\n        super().__init__()\n        self.model_dim = model_dim\n        self.num_layers = num_layers\n        self.joint_count = joint_count\n        self.time_embed_dim = time_embed_dim\n        self.kinematic_encoder = KinematicChainEncoder(model_dim)\n        self.time_embed = nn.Sequential(nn.Linear(time_embed_dim, model_dim), nn.SiLU(), nn.Linear(model_dim, model_dim))\n        self.input_proj = nn.Linear(per_joint_dim + 3 + 6 + 3 + 3, model_dim)\n        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=4, dim_feedforward=model_dim * 2, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)\n        self.spatial_transformer = nn.TransformerEncoder(encoder_layer, num_layers, enable_nested_tensor=False)\n        self.noise_pred = nn.Sequential(nn.Linear(model_dim, model_dim), nn.GELU(), nn.Linear(model_dim, 3))\n        self.null_prev_frame = nn.Parameter(torch.zeros(1, 22, 12))\n        self.null_progress = nn.Parameter(torch.zeros(1, 1, 1))\n\n    def _sinusoidal_time_embedding(self, t: torch.Tensor, max_positions=10000) -> torch.Tensor:\n        half_dim = self.time_embed_dim // 2\n        freqs = torch.exp(-torch.log(torch.tensor(max_positions)) / (half_dim - 1) * torch.arange(half_dim, device=t.device))\n        args = t.unsqueeze(-1) * freqs.unsqueeze(0)\n        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)\n        if self.time_embed_dim % 2 != 0:\n            emb = nn.functional.pad(emb, (0, 1), mode='constant')\n        return emb\n\n    def forward(self, history_features: torch.Tensor, noise_level: torch.Tensor, noisy_target: Optional[torch.Tensor]=None, prev_frame_features: Optional[torch.Tensor]=None, temporal_progress: Optional[torch.Tensor]=None) -> torch.Tensor:\n        B, joint_count, _ = history_features.shape\n        assert B == noise_level.shape[0], f'noise level {noise_level.shape} does not match batch size {B}'\n        if temporal_progress is not None:\n            assert B == temporal_progress.shape[0], f'temporal progress {temporal_progress.shape} does not match batch size {B}'\n        if prev_frame_features is not None:\n            assert B == prev_frame_features.shape[0], f'prev frame features {prev_frame_features.shape} does not match batch size {B}'\n        if noisy_target is not None:\n            assert B == noisy_target.shape[0], f'noisy target diffs {noisy_target.shape} does not match batch size {B}'\n        if noisy_target is None:\n            noisy_target = torch.randn((B, joint_count, 3), device=history_features.device)\n        if prev_frame_features is None:\n            prev_frame_features = self.null_prev_frame.expand(B, joint_count, 12)\n        x = torch.cat([history_features, prev_frame_features, noisy_target], dim=-1)\n        x = self.input_proj(x)\n        time_embed = self._sinusoidal_time_embedding(noise_level)\n        time_bias = self.time_embed(time_embed)\n        x = x + time_bias.unsqueeze(1)\n        joint_ids = torch.arange(joint_count, device=x.device)\n        kinematic_bias = self.kinematic_encoder(joint_ids)\n        x = x + kinematic_bias.unsqueeze(0)\n        x = self.spatial_transformer(x)\n        pred_noise: torch.Tensor = self.noise_pred(x)\n        return pred_noise\n\nclass HumanMotionGenerator(nn.Module):\n\n    def __init__(self, encoder: MotionHistoryEncoder, predictor: FlowMatchingPredictor) -> None:\n        super().__init__()\n        self.encoder = encoder\n        self.predictor = predictor\n    '\\n    Updated generate_sequence() methods for HumanMotionGenerator class\\n    Returns GLOBAL JOINT POSITIONS instead of feature vectors\\n\\n    Add these methods to the HumanMotionGenerator class in models.py\\n    '\n\n    def generate_sequence(self, text: Union[str, List[str], torch.Tensor], num_frames: int=200, num_steps: int=10, guidance_scale: float=2.5, input_features: Optional[torch.Tensor]=None, total_duration: Optional[torch.Tensor]=None, dataset_type: str='t2m') -> torch.Tensor:\n        self.eval()\n        with torch.no_grad():\n            if isinstance(text, torch.Tensor):\n                B = text.shape[0]\n            else:\n                B = 1\n            device = next(self.parameters()).device\n            T_hist = 15\n            if input_features is None:\n                history = self.encoder.null_history.expand(B, T_hist, -1).clone()\n            elif input_features.shape[1] >= T_hist:\n                history = input_features[:, -T_hist:, :]\n            else:\n                pad_size = T_hist - input_features.shape[1]\n                null_pad = self.encoder.null_history.expand(B, pad_size, -1).clone()\n                history = torch.cat([null_pad, input_features], dim=1)\n            extractor = IncrementalFeatureExtractor(dataset_type=dataset_type, feet_thre=0.002, device=device)\n            joint_sequence = []\n            init_frame = history[:, -1, :]\n            current_joints_global = features_to_positions(init_frame, dataset_type=dataset_type)\n            extractor.initialize(current_joints_global)\n            for frame_idx in range(num_frames):\n                t_prog = torch.full((B,), frame_idx / num_frames, device=device)\n                context_cond = self.encoder(batch_size=B, text=text, input_features=history)\n                context_uncond = self.encoder(batch_size=B, text=None, input_features=history)\n                last_frame = history[:, -1, :]\n                ric_pos_21 = last_frame[:, 4:67].reshape(B, 21, 3)\n                root_pos = torch.zeros((B, 1, 3), device=device, dtype=last_frame.dtype)\n                prev_pos_ric = torch.cat([root_pos, ric_pos_21], dim=1)\n                prev_rot6d = last_frame[:, 67:193].reshape(B, 21, 6)\n                root_rot = torch.zeros((B, 1, 6), device=device, dtype=last_frame.dtype)\n                root_rot[:, 0, 0] = 1.0\n                root_rot[:, 0, 4] = 1.0\n                prev_rot6d = torch.cat([root_rot, prev_rot6d], dim=1)\n                prev_v = last_frame[:, 193:259].reshape(B, 22, 3)\n                prev_frame_features = torch.cat([prev_pos_ric, prev_rot6d, prev_v], dim=-1)\n                x_t = torch.randn((B, 22, 3), device=device)\n                dt = 1.0 / num_steps\n                for step in range(num_steps):\n                    t = torch.full((B,), step * dt, device=device)\n                    v_cond = self.predictor(history_features=context_cond, noise_level=t, noisy_target=x_t, prev_frame_features=prev_frame_features)\n                    v_uncond = self.predictor(history_features=context_uncond, noise_level=t, noisy_target=x_t, prev_frame_features=prev_frame_features)\n                    v_t = v_uncond + guidance_scale * (v_cond - v_uncond)\n                    x_t = x_t + v_t * dt\n                new_joints_global = current_joints_global + x_t\n                current_joints_global = new_joints_global.clone()\n                joint_sequence.append(new_joints_global.cpu())\n                new_frame_features = extractor.process_frame(new_joints_global)\n                history = torch.cat([history[:, 1:, :], new_frame_features.unsqueeze(1)], dim=1)\n                if (frame_idx + 1) % 50 == 0:\n                    print(f'Generated {frame_idx + 1}/{num_frames} frames')\n            joint_positions = torch.stack(joint_sequence, dim=1).to(device)\n            return joint_positions\n\n    @classmethod\n    def load_from_checkpoint(cls, checkpoint_path: Union[str, Path], config: Config, device: str='cpu') -> 'HumanMotionGenerator':\n        print(f'Loading checkpoint from {checkpoint_path}...')\n        checkpoint = torch.load(checkpoint_path, map_location=device)\n        encoder = MotionHistoryEncoder(frame_feature_dim=config.motion_dim, text_embedding_dim=config.text_embedding_dim, joint_feature_projection_dim=config.joint_feature_projection_dim, text_projection_dim=config.text_projection_dim, per_joint_out_dim=config.per_joint_out_dim, joint_count=config.num_joints, model_dim=config.model_dim, num_layers=config.num_encoder_layers, bidirectional=config.bidirectional_gru)\n        predictor = FlowMatchingPredictor(per_joint_dim=config.per_joint_out_dim, model_dim=config.model_dim, num_layers=config.num_flow_layers, joint_count=config.num_joints)\n        if 'ema_mhe' in checkpoint and 'ema_fmp' in checkpoint:\n            print('Loading EMA weights for generation...')\n            encoder.load_state_dict(checkpoint['ema_mhe'])\n            predictor.load_state_dict(checkpoint['ema_fmp'])\n        else:\n            print('Loading standard weights (EMA not found)...')\n            encoder.load_state_dict(checkpoint['motion_history_encoder'])\n            predictor.load_state_dict(checkpoint['flow_predictor'])\n        encoder.to(device)\n        predictor.to(device)\n        encoder.eval()\n        predictor.eval()\n        return cls(encoder, predictor)",
        'requirements.txt': '# Core ML dependencies\ntorch\ntorchvision\nnumpy\nscipy\ntransformers\n\n# Data processing\npandas\n\n# Visualization\nmatplotlib\nseaborn\n\n# Utilities\ntqdm\ngdown\n\n# Text Encoding\nftfy\nregex\n',
        'utils/__init__.py': '# Utils module for motion generation project\n',
        'utils/utils.py': "from .dataset import Text2MotionDataset, create_dataloader, load_sample\nfrom .motion_utils import DATASET_CONFIGS, get_dataset_config, features_to_positions, preprocess_sequence, get_feature_subset, IncrementalFeatureExtractor\nfeature_to_joints = features_to_positions\nfrom .visualization import plot_3d_motion, visualize_motion, compare_motions\nfrom .bvh_utils import joints_to_bvh, save_bvh, save_joints, validate_bvh\nfrom .metrics import compute_metrics\n__all__ = ['Text2MotionDataset', 'create_dataloader', 'load_sample', 'DATASET_CONFIGS', 'get_dataset_config', 'features_to_positions', 'feature_to_joints', 'preprocess_sequence', 'get_feature_subset', 'IncrementalFeatureExtractor', 'plot_3d_motion', 'visualize_motion', 'compare_motions', 'joints_to_bvh', 'save_bvh', 'save_joints', 'validate_bvh', 'compute_metrics']",
        'utils/dataset.py': "import torch\nimport numpy as np\nfrom os.path import join as pjoin\nimport random\nfrom tqdm import tqdm\nfrom torch.utils.data import Dataset, DataLoader\nfrom pathlib import Path\nfrom typing import List, Dict, Any, Optional, Tuple\nfrom config import Config\nfrom .motion_utils import get_feature_vec_subset\n\nclass Text2MotionDataset(Dataset):\n\n    def __init__(self, config: Config, mean: np.ndarray, std: np.ndarray, split: str='train', feature_dims: tuple[slice, ...] | None=None):\n        self.config = config\n        self.feature_dims = feature_dims if feature_dims is not None else config.feature_dims\n        self.max_length = 20\n        self.pointer = 0\n        self.max_motion_length = config.max_motion_length\n        min_motion_len = 40 if config.dataset_name == 't2m' else 24\n        motion_dir = config.dataset_path / 'new_joint_vecs'\n        joints_dir = config.dataset_path / 'new_joints'\n        text_dir = config.dataset_path / 'texts'\n        split_file = config.dataset_path / f'{split}.txt'\n        data_dict = {}\n        id_list = []\n        with open(str(split_file), 'r', encoding='utf-8') as f:\n            for line in f.readlines():\n                id_list.append(line.strip())\n        new_name_list = []\n        length_list = []\n        for name in tqdm(id_list):\n            try:\n                motion = np.load(pjoin(str(motion_dir), name + '.npy'))\n                joints = np.load(pjoin(str(joints_dir), name + '.npy'))\n                if len(motion) < min_motion_len or len(motion) >= 200:\n                    continue\n                text_data = []\n                flag = False\n                with open(pjoin(str(text_dir), name + '.txt'), 'r', encoding='utf-8') as f:\n                    for line in f.readlines():\n                        text_dict: Dict[str, Optional[Any]] = {}\n                        line_split = line.strip().split('#')\n                        caption = line_split[0]\n                        tokens = line_split[1].split(' ')\n                        f_tag = float(line_split[2])\n                        to_tag = float(line_split[3])\n                        f_tag = 0.0 if np.isnan(f_tag) else f_tag\n                        to_tag = 0.0 if np.isnan(to_tag) else to_tag\n                        text_dict['caption'] = caption\n                        text_dict['tokens'] = tokens\n                        if f_tag == 0.0 and to_tag == 0.0:\n                            flag = True\n                            text_data.append(text_dict)\n                        else:\n                            try:\n                                n_motion = motion[int(f_tag * 20):int(to_tag * 20)]\n                                if len(n_motion) < min_motion_len or len(n_motion) >= 200:\n                                    continue\n                                new_name = random.choice('ABCDEFGHIJKLMNOPQRSTUVW') + '_' + name\n                                while new_name in data_dict:\n                                    new_name = random.choice('ABCDEFGHIJKLMNOPQRSTUVW') + '_' + name\n                                n_joints = joints[int(f_tag * 20):int(to_tag * 20)]\n                                data_dict[new_name] = {'motion': n_motion, 'joints': n_joints, 'length': len(n_motion), 'text': [text_dict]}\n                                new_name_list.append(new_name)\n                                length_list.append(len(n_motion))\n                            except:\n                                print(line_split)\n                                print(line_split[2], line_split[3], f_tag, to_tag, name)\n                if flag:\n                    data_dict[name] = {'motion': motion, 'joints': joints, 'length': len(motion), 'text': text_data}\n                    new_name_list.append(name)\n                    length_list.append(len(motion))\n            except Exception as e:\n                pass\n        name_list, length_list = (new_name_list, length_list)\n        self.mean = torch.from_numpy(mean).float()\n        self.std = torch.from_numpy(std).float()\n        self.length_arr = np.array(length_list)\n        self.data_dict = data_dict\n        self.name_list = name_list\n        self.text_cache_path = config.dataset_path / 'text_embeddings_cache.pt'\n        self.text_cache: Dict[str, torch.Tensor] = {}\n        if self.text_cache_path.exists():\n            print(f'Loading text embedding cache from {self.text_cache_path}...')\n            self.text_cache = torch.load(self.text_cache_path)\n        all_captions = set()\n        for key, data in self.data_dict.items():\n            for text_item in data['text']:\n                all_captions.add(text_item['caption'])\n        missing_captions = [cap for cap in all_captions if cap not in self.text_cache]\n        if missing_captions:\n            print(f'Computed {len(self.text_cache)}/{len(all_captions)} embeddings. Computing {len(missing_captions)} missing...')\n            from .text_encoder import CLIPEncoder\n            clip_encoder = CLIPEncoder(model_name='openai/clip-vit-base-patch32')\n            clip_encoder.to(config.device)\n            batch_size = 32\n            for i in tqdm(range(0, len(missing_captions), batch_size), desc='Encoding Texts'):\n                batch_caps = missing_captions[i:i + batch_size]\n                with torch.no_grad():\n                    embeddings = clip_encoder(batch_caps).cpu()\n                for cap, emb in zip(batch_caps, embeddings):\n                    self.text_cache[cap] = emb\n            print(f'Saving updated cache to {self.text_cache_path}...')\n            torch.save(self.text_cache, self.text_cache_path)\n            del clip_encoder\n            torch.cuda.empty_cache()\n        else:\n            print('All text embeddings are cached.')\n\n    def inv_transform(self, data):\n        return data * self.std + self.mean\n\n    def __len__(self):\n        return len(self.data_dict) - self.pointer\n    '\\n    FINAL CORRECT __getitem__ implementation\\n    This is the ONLY version that works - replace everything else\\n    '\n\n    def __getitem__(self, item):\n        idx = self.pointer + item\n        data = self.data_dict[self.name_list[idx]]\n        motion = data['motion']\n        joints = data['joints']\n        original_length = data['length']\n        text_list = data['text']\n        text_data = random.choice(text_list)\n        caption = text_data['caption']\n        motion = torch.from_numpy(motion.copy()).float()\n        joints = torch.from_numpy(joints.copy()).float()\n        motion = (motion - self.mean) / self.std\n        m_length = original_length\n        if self.config.unit_length < 10:\n            coin2 = np.random.choice(['single', 'single', 'double'])\n        else:\n            coin2 = 'single'\n        if coin2 == 'double':\n            m_length = (m_length // self.config.unit_length - 1) * self.config.unit_length\n        else:\n            m_length = m_length // self.config.unit_length * self.config.unit_length\n        m_length = min(m_length, len(motion))\n        m_length = max(1, m_length)\n        motion = motion[:m_length]\n        joints = joints[:m_length]\n        target_len = self.max_motion_length\n        current_len = len(motion)\n        if current_len < target_len:\n            pad_size = target_len - current_len\n            motion = torch.cat([motion, torch.zeros(pad_size, motion.shape[1], dtype=motion.dtype, device=motion.device)], dim=0)\n            joints = torch.cat([joints, torch.zeros(pad_size, joints.shape[1], joints.shape[2], dtype=joints.dtype, device=joints.device)], dim=0)\n        elif current_len > target_len:\n            motion = motion[:target_len]\n            joints = joints[:target_len]\n        expected_motion_dim = int(self.mean.shape[0])\n        assert motion.shape[0] == target_len, f'Motion shape[0]={motion.shape[0]}, expected {target_len}'\n        assert motion.shape[1] == expected_motion_dim, f'Motion shape[1]={motion.shape[1]}, expected {expected_motion_dim}'\n        assert joints.shape[0] == target_len, f'Joints shape[0]={joints.shape[0]}, expected {target_len}'\n        history_features = get_feature_vec_subset(motion, self.feature_dims)\n        if isinstance(history_features, np.ndarray):\n            history_features = torch.from_numpy(history_features).float()\n        else:\n            history_features = history_features.float()\n        assert history_features.shape[0] == target_len, f'history_features shape[0]={history_features.shape[0]}, expected {target_len}'\n        text_embedding = self.text_cache[caption]\n        if isinstance(text_embedding, np.ndarray):\n            text_embedding = torch.from_numpy(text_embedding).float()\n        else:\n            text_embedding = text_embedding.float()\n        return (caption, history_features, motion, joints, m_length, text_embedding)\n\n    def reset_min_len(self, length):\n        assert length <= self.max_motion_length\n        self.pointer = np.searchsorted(self.length_arr, length)\n        print('Pointer Pointing at %d' % self.pointer)\nfrom typing import List, Dict, Any\n\ndef text2motion_collate_fn(batch: List[Tuple[str, torch.Tensor, torch.Tensor, torch.Tensor, int, torch.Tensor]]) -> Dict[str, Any]:\n    captions = [b[0] for b in batch]\n    cond_feats_list = [b[1] for b in batch]\n    motions_list = [b[2] for b in batch]\n    joints_list = [b[3] for b in batch]\n    lengths = [b[4] for b in batch]\n    text_embs_list = [b[5] for b in batch]\n\n    def to_tensor(x):\n        if isinstance(x, np.ndarray):\n            return torch.from_numpy(x).float()\n        elif isinstance(x, torch.Tensor):\n            return x.float()\n        else:\n            return torch.tensor(x).float()\n    cond_feats_list = [to_tensor(x) for x in cond_feats_list]\n    motions_list = [to_tensor(x) for x in motions_list]\n    joints_list = [to_tensor(x) for x in joints_list]\n    text_embs_list = [to_tensor(x) for x in text_embs_list]\n    cond_feature_batch = torch.stack(cond_feats_list, dim=0)\n    motion_batch = torch.stack(motions_list, dim=0)\n    joints_batch = torch.stack(joints_list, dim=0)\n    length_batch = torch.tensor(lengths, dtype=torch.long)\n    text_emb_batch = torch.stack(text_embs_list, dim=0)\n    return {'captions': captions, 'history_features': cond_feature_batch, 'motion': motion_batch, 'joints': joints_batch, 'lengths': length_batch, 'text_clip': text_emb_batch}\n\ndef create_dataloader(config: Config, split: str='train', shuffle: bool=True) -> DataLoader:\n    mean_path = config.dataset_path / 'Mean.npy'\n    std_path = config.dataset_path / 'Std.npy'\n    if not mean_path.exists() or not std_path.exists():\n        raise FileNotFoundError(f'Mean.npy and/or Std.npy not found in {config.dataset_path}. Please ensure Mean.npy and Std.npy exist in the dataset directory.')\n    mean = np.load(mean_path)\n    std = np.load(std_path)\n    dataset_obj = Text2MotionDataset(config, mean, std, split, feature_dims=config.feature_dims)\n    return DataLoader(dataset_obj, batch_size=config.batch_size, shuffle=shuffle, num_workers=0, pin_memory=config.pin_memory, collate_fn=text2motion_collate_fn)\n\ndef load_sample(dataset_path: Path, file_id: str) -> Dict[str, Optional[Any]]:\n    features_path = dataset_path / 'new_joint_vecs' / f'{file_id}.npy'\n    joints_path = dataset_path / 'new_joints' / f'{file_id}.npy'\n    text_path = dataset_path / 'texts' / f'{file_id}.txt'\n    data: Dict[str, Optional[Any]] = {'file_id': file_id}\n    if features_path.exists():\n        data['features'] = np.load(features_path)\n    else:\n        print(f'Warning: Features not found for {file_id}')\n        data['features'] = None\n    if joints_path.exists():\n        data['joints'] = np.load(joints_path)\n    else:\n        print(f'Warning: Joints not found for {file_id}')\n        data['joints'] = None\n    if text_path.exists():\n        with open(text_path, 'r') as f:\n            descriptions = [line.strip().split('#')[0] for line in f.readlines()]\n            data['text'] = descriptions[0] if descriptions else ''\n    else:\n        data['text'] = ''\n    return data",
        'utils/motion_utils.py': "import torch\nimport numpy as np\nfrom typing import List, Tuple, Dict, Any, Optional\nfrom .skeleton import Skeleton\nfrom .quaternion import qrot, qinv, qmul, quaternion_to_cont6d, cont6d_to_matrix, cont6d_to_quaternion\nT2M_RAW_OFFSETS = torch.tensor([[0, 0, 0], [1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1], [0, 1, 0], [1, 0, 0], [-1, 0, 0], [0, 0, 1], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0]], dtype=torch.float32)\nT2M_KINEMATIC_CHAIN = [[0, 2, 5, 8, 11], [0, 1, 4, 7, 10], [0, 3, 6, 9, 12, 15], [9, 14, 17, 19, 21], [9, 13, 16, 18, 20]]\nt2m_kinematic_chain = T2M_KINEMATIC_CHAIN\nDATASET_CONFIGS = {'t2m': {'name': 'HumanML3D', 'num_joints': 22, 'feature_dim': 271, 'raw_offsets': T2M_RAW_OFFSETS, 'kinematic_chain': T2M_KINEMATIC_CHAIN, 'face_joint_indx': [2, 1, 17, 16], 'fid_r': [8, 11], 'fid_l': [7, 10]}}\n\ndef get_dataset_config(dataset_type: str='t2m') -> Dict[str, Any]:\n    if dataset_type not in DATASET_CONFIGS:\n        raise ValueError(f'Unknown dataset_type: {dataset_type}. Available: {list(DATASET_CONFIGS.keys())}')\n    return DATASET_CONFIGS[dataset_type]\nFEATURE_SLICES = {'global_root_pos': slice(0, 3), 'ric_positions': slice(3, 69), 'rotations_6d': slice(69, 201), 'local_velocities': slice(201, 267), 'foot_contacts': slice(267, 271)}\n\ndef _compute_ik(positions: torch.Tensor, raw_offsets: torch.Tensor, kinematic_chain: List[List[int]], face_joint_indx: List[int]) -> torch.Tensor:\n    batch_shape = positions.shape[:-2]\n    device = positions.device\n    dtype = positions.dtype\n    positions_flat = positions.reshape(-1, 22, 3)\n    B = positions_flat.shape[0]\n    l_hip, r_hip, sdr_r, sdr_l = face_joint_indx\n    across1 = positions_flat[:, r_hip] - positions_flat[:, l_hip]\n    across2 = positions_flat[:, sdr_r] - positions_flat[:, sdr_l]\n    across = across1 + across2\n    across = across / (torch.norm(across, dim=-1, keepdim=True) + 1e-10)\n    forward = torch.cross(torch.tensor([[0, 1, 0]], device=device, dtype=dtype).expand(B, -1), across, dim=-1)\n    forward = forward / (torch.norm(forward, dim=-1, keepdim=True) + 1e-10)\n    target = torch.tensor([[0, 0, 1]], device=device, dtype=dtype).expand(B, -1)\n    root_quat = _qbetween(forward, target)\n    quaternions = torch.zeros(B, 22, 4, device=device, dtype=dtype)\n    quaternions[:, 0] = root_quat\n    offsets = raw_offsets.unsqueeze(0).expand(B, -1, -1)\n    for chain in kinematic_chain:\n        R = root_quat\n        for i in range(len(chain) - 1):\n            parent_idx = chain[i]\n            child_idx = chain[i + 1]\n            u = offsets[:, child_idx]\n            v = positions_flat[:, child_idx] - positions_flat[:, parent_idx]\n            v = v / (torch.norm(v, dim=-1, keepdim=True) + 1e-10)\n            rot_u_v = _qbetween(u, v)\n            R_loc = qmul(qinv(R), rot_u_v)\n            quaternions[:, child_idx] = R_loc\n            R = qmul(R, R_loc)\n    return quaternions.reshape(batch_shape + (22, 4))\n\ndef _qbetween(v0: torch.Tensor, v1: torch.Tensor) -> torch.Tensor:\n    v0 = v0 / (torch.norm(v0, dim=-1, keepdim=True) + 1e-10)\n    v1 = v1 / (torch.norm(v1, dim=-1, keepdim=True) + 1e-10)\n    dot = (v0 * v1).sum(dim=-1, keepdim=True)\n    cross = torch.cross(v0, v1, dim=-1)\n    w = 1.0 + dot\n    q = torch.cat([w, cross], dim=-1)\n    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)\n    return q\n\ndef _forward_kinematics(rotations_6d: torch.Tensor, root_pos: torch.Tensor, offsets: torch.Tensor, kinematic_chain: List[List[int]]) -> torch.Tensor:\n    batch_shape = rotations_6d.shape[:-2]\n    device = rotations_6d.device\n    dtype = rotations_6d.dtype\n    rotations_flat = rotations_6d.reshape(-1, 22, 6)\n    root_pos_flat = root_pos.reshape(-1, 3)\n    B = rotations_flat.shape[0]\n    positions = torch.zeros(B, 22, 3, device=device, dtype=dtype)\n    positions[:, 0] = root_pos_flat\n    offsets_expanded = offsets.unsqueeze(0).expand(B, -1, -1)\n    for chain in kinematic_chain:\n        matR = cont6d_to_matrix(rotations_flat[:, 0])\n        for i in range(1, len(chain)):\n            child_idx = chain[i]\n            parent_idx = chain[i - 1]\n            child_rot = cont6d_to_matrix(rotations_flat[:, child_idx])\n            matR = torch.bmm(matR, child_rot)\n            offset_vec = offsets_expanded[:, child_idx].unsqueeze(-1)\n            positions[:, child_idx] = torch.bmm(matR, offset_vec).squeeze(-1) + positions[:, parent_idx]\n    return positions.reshape(batch_shape + (22, 3))\n\ndef preprocess_sequence(positions: torch.Tensor, dataset_type: str='t2m', feet_thre: float=0.002) -> torch.Tensor:\n    config = get_dataset_config(dataset_type)\n    raw_offsets = config['raw_offsets']\n    kinematic_chain = config['kinematic_chain']\n    face_joint_indx = config['face_joint_indx']\n    fid_r = config['fid_r']\n    fid_l = config['fid_l']\n    device = positions.device\n    dtype = positions.dtype\n    if positions.ndim == 4:\n        B, N, J, _ = positions.shape\n        features_batch = []\n        for b in range(B):\n            pos_b = positions[b]\n            feat_b = preprocess_sequence(pos_b, dataset_type, feet_thre)\n            features_batch.append(feat_b)\n        return torch.stack(features_batch, dim=0)\n    N = positions.shape[0]\n    global_root_pos = positions[:, 0].clone()\n    quaternions = _compute_ik(positions, raw_offsets, kinematic_chain, face_joint_indx)\n    root_quat = quaternions[:, 0].clone()\n    ric = positions - positions[:, 0:1, :]\n    ric = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)\n    rotations_6d = quaternion_to_cont6d(quaternions)\n    local_vel = torch.zeros(N, 22, 3, device=device, dtype=dtype)\n    if N > 1:\n        local_vel[1:] = qrot(root_quat[1:].unsqueeze(1).expand(-1, 22, -1), positions[1:] - positions[:-1])\n    feet_l = torch.zeros(N, 2, device=device, dtype=dtype)\n    feet_r = torch.zeros(N, 2, device=device, dtype=dtype)\n    if N > 1:\n        vel_l = positions[1:, fid_l] - positions[:-1, fid_l]\n        vel_r = positions[1:, fid_r] - positions[:-1, fid_r]\n        feet_l[1:] = (torch.sum(vel_l ** 2, dim=-1) < feet_thre).float()\n        feet_r[1:] = (torch.sum(vel_r ** 2, dim=-1) < feet_thre).float()\n    features = torch.cat([global_root_pos, ric.reshape(N, -1), rotations_6d.reshape(N, -1), local_vel.reshape(N, -1), feet_l, feet_r], dim=-1)\n    return features\n\ndef features_to_positions(features: torch.Tensor, dataset_type: str='t2m') -> torch.Tensor:\n    global_root_pos = features[..., 0:3]\n    ric = features[..., 3:69].reshape(features.shape[:-1] + (22, 3))\n    rotations_6d = features[..., 69:201].reshape(features.shape[:-1] + (22, 6))\n    root_quat = cont6d_to_quaternion(rotations_6d[..., 0, :])\n    root_quat_expanded = root_quat.unsqueeze(-2).expand(root_quat.shape[:-1] + (22, -1))\n    positions = global_root_pos.unsqueeze(-2) + qrot(qinv(root_quat_expanded), ric)\n    return positions\n\nclass IncrementalFeatureExtractor:\n\n    def __init__(self, dataset_type: str='t2m', feet_thre: float=0.002, device: torch.device=torch.device('cpu'), dtype: torch.dtype=torch.float32):\n        config = get_dataset_config(dataset_type)\n        self.raw_offsets = config['raw_offsets'].to(device).to(dtype)\n        self.kinematic_chain = config['kinematic_chain']\n        self.face_joint_indx = config['face_joint_indx']\n        self.fid_r = config['fid_r']\n        self.fid_l = config['fid_l']\n        self.feet_thre = feet_thre\n        self.device = device\n        self.dtype = dtype\n        self.prev_positions: Optional[torch.Tensor] = None\n        self.is_initialized = False\n\n    def initialize(self, initial_positions: torch.Tensor) -> torch.Tensor:\n        initial_positions = initial_positions.to(self.device).to(self.dtype)\n        B = initial_positions.shape[0]\n        self.prev_positions = initial_positions.clone()\n        quaternions = _compute_ik(initial_positions, self.raw_offsets, self.kinematic_chain, self.face_joint_indx)\n        rotations_6d = quaternion_to_cont6d(quaternions)\n        root_pos = initial_positions[:, 0]\n        fk_positions = _forward_kinematics(rotations_6d, root_pos, self.raw_offsets, self.kinematic_chain)\n        self.prev_fk_positions = fk_positions\n        self.is_initialized = True\n        return torch.zeros(B, 271, device=self.device, dtype=self.dtype)\n\n    def process_frame(self, positions: torch.Tensor) -> torch.Tensor:\n        positions = positions.to(self.device).to(self.dtype)\n        if not self.is_initialized:\n            return self.initialize(positions)\n        B = positions.shape[0]\n        global_root_pos = positions[:, 0]\n        quaternions = _compute_ik(positions, self.raw_offsets, self.kinematic_chain, self.face_joint_indx)\n        root_quat = quaternions[:, 0]\n        rotations_6d = quaternion_to_cont6d(quaternions)\n        fk_positions = _forward_kinematics(rotations_6d, global_root_pos, self.raw_offsets, self.kinematic_chain)\n        ric = fk_positions - fk_positions[:, 0:1]\n        ric = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)\n        local_vel = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), positions - self.prev_positions)\n        foot_vel = positions - self.prev_positions\n        feet_l = (torch.sum(foot_vel[:, self.fid_l] ** 2, dim=-1) < self.feet_thre).float()\n        feet_r = (torch.sum(foot_vel[:, self.fid_r] ** 2, dim=-1) < self.feet_thre).float()\n        self.prev_positions = positions.clone()\n        features = torch.cat([global_root_pos, ric.reshape(B, -1), rotations_6d.reshape(B, -1), local_vel.reshape(B, -1), feet_l, feet_r], dim=-1)\n        return features\n\n    def reset(self):\n        self.prev_positions = None\n        self.prev_fk_positions = None\n        self.is_initialized = False\n\ndef get_feature_subset(features: torch.Tensor, subset_names: List[str]) -> torch.Tensor:\n    subsets = []\n    for name in subset_names:\n        if name not in FEATURE_SLICES:\n            raise ValueError(f'Unknown feature: {name}. Available: {list(FEATURE_SLICES.keys())}')\n        subsets.append(features[..., FEATURE_SLICES[name]])\n    return torch.cat(subsets, dim=-1)\n\ndef get_feature_vec_subset(features: torch.Tensor, dimensions: Tuple[slice, ...]) -> torch.Tensor:\n    subset_list = []\n    if isinstance(features, torch.Tensor):\n        for dim in dimensions:\n            subset_list.append(features[:, dim])\n        return torch.cat(subset_list, dim=-1)\n    for dim in dimensions:\n        subset_list.append(features[:, dim])\n    return np.concatenate(subset_list, axis=-1)",
        'utils/visualization.py': "import numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib.animation import FuncAnimation\nfrom pathlib import Path\nfrom typing import Optional, Any\nfrom .motion_utils import t2m_kinematic_chain\n\ndef plot_3d_motion(motion: np.ndarray, fps: float=20, radius: float=1.0, title: str='Motion Visualization', follow_root: bool=False) -> FuncAnimation:\n    fig = plt.figure(figsize=(8, 8))\n    ax = fig.add_subplot(111, projection='3d')\n    ax.view_init(elev=15, azim=-70)\n    colors = ['#2980b9', '#c0392b', '#27ae60', '#f39c12', '#8e44ad']\n    lines = [ax.plot([], [], [], color=colors[i % len(colors)], marker='o', ms=2, lw=2)[0] for i in range(len(t2m_kinematic_chain))]\n    ax.set_xlabel('X (Side)')\n    ax.set_ylabel('Z (Forward)')\n    ax.set_zlabel('Y (Height)')\n    ax.set_title(title)\n    pos_min = motion.min(axis=(0, 1))\n    pos_max = motion.max(axis=(0, 1))\n\n    def update(frame):\n        root = motion[frame, 0, :]\n        if follow_root:\n            ax.set_xlim3d([root[0] - radius, root[0] + radius])\n            ax.set_ylim3d([root[2] - radius, root[2] + radius])\n            ax.set_zlim3d([pos_min[1], pos_max[1] + radius * 0.5])\n        else:\n            ax.set_xlim3d([pos_min[0] - radius, pos_max[0] + radius])\n            ax.set_ylim3d([pos_min[2] - radius, pos_max[2] + radius])\n            ax.set_zlim3d([pos_min[1], pos_max[1] + radius * 0.5])\n        for i, c_indices in enumerate(t2m_kinematic_chain):\n            joints = motion[frame, c_indices, :]\n            lines[i].set_data(joints[:, 0], joints[:, 2])\n            lines[i].set_3d_properties(joints[:, 1])\n        return lines\n    ani = FuncAnimation(fig, update, frames=len(motion), interval=1000 / fps, blit=False)\n    plt.close()\n    return ani\n\ndef visualize_motion(joint_positions: np.ndarray, ground_truth: Optional[np.ndarray]=None, title: str='Motion Visualization', save_path: Optional[Path]=None, fps: float=20, skip_frames: int=1, notebook: bool=True) -> Any:\n    fps = fps / skip_frames\n    ani = plot_3d_motion(joint_positions[::skip_frames], fps=fps, title=title)\n    if save_path:\n        save_path.parent.mkdir(parents=True, exist_ok=True)\n        ani.save(str(save_path), writer='ffmpeg', fps=int(fps))\n        print(f'Saved animation to {save_path}')\n    if notebook:\n        from IPython.display import HTML\n        return HTML(ani.to_html5_video())\n    return ani\n\ndef compare_motions(generated_joints: np.ndarray, ground_truth_joints: np.ndarray, save_path: Optional[Path]=None) -> None:\n    visualize_motion(generated_joints, ground_truth=ground_truth_joints, title='Generated vs Ground Truth', save_path=save_path)",
        'utils/bvh_utils.py': "import numpy as np\nfrom pathlib import Path\nfrom typing import Dict, Any, Optional\n\ndef _get_default_skeleton_hierarchy() -> Dict:\n    parents = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19]\n    joint_names = ['pelvis', 'left_hip', 'right_hip', 'spine1', 'left_knee', 'right_knee', 'spine2', 'left_ankle', 'right_ankle', 'spine3', 'left_foot', 'right_foot', 'neck', 'left_collar', 'right_collar', 'head', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist']\n    hierarchy = {}\n    for i, name in enumerate(joint_names):\n        p_idx = parents[i]\n        p_name = joint_names[p_idx] if p_idx != -1 else 'root'\n        if p_name not in hierarchy:\n            hierarchy[p_name] = {'children': []}\n        hierarchy[p_name]['children'].append(name)\n        if name not in hierarchy:\n            hierarchy[name] = {'children': []}\n    return hierarchy\n\ndef joints_to_bvh(joint_positions: np.ndarray, fps: int=20, skeleton_template: Optional[Dict]=None) -> Dict[str, Any]:\n    nframe, num_joints, _ = joint_positions.shape\n    bvh_data = {'hierarchy': skeleton_template or _get_default_skeleton_hierarchy(), 'motion': {'frames': nframe, 'fps': fps, 'data': joint_positions.tolist()}}\n    print(f'TODO: Implement proper joints_to_bvh conversion')\n    print(f'Input: {joint_positions.shape} -> BVH format')\n    return bvh_data\n\ndef save_bvh(bvh_data: Dict[str, Any], output_path: Path) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    with open(output_path, 'w') as f:\n        f.write('HIERARCHY\\n')\n        f.write('ROOT root\\n')\n        f.write('{\\n')\n        f.write('  OFFSET 0.0 0.0 0.0\\n')\n        f.write('  CHANNELS 6 Xposition Yposition Zposition Zrotation Xrotation Yrotation\\n')\n        f.write('}\\n')\n        f.write('MOTION\\n')\n        f.write(f'Frames: {bvh_data['motion']['frames']}\\n')\n        f.write(f'Frame Time: {1.0 / bvh_data['motion']['fps']:.6f}\\n')\n    print(f'TODO: Implement complete BVH file writing')\n    print(f'Saved BVH to {output_path}')\n\ndef save_joints(joint_positions: np.ndarray, output_path: Path) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    np.save(output_path, joint_positions)\n    print(f'Saved joints to {output_path}')\n\ndef validate_bvh(bvh_path: Path) -> bool:\n    if not bvh_path.exists():\n        return False\n    try:\n        with open(bvh_path, 'r') as f:\n            content = f.read()\n            if 'HIERARCHY' in content and 'MOTION' in content:\n                return True\n    except Exception:\n        return False\n    return False",
        'utils/metrics.py': "import numpy as np\nfrom typing import List, Dict\n\ndef compute_metrics(generated_joints: List[np.ndarray], ground_truth_joints: List[np.ndarray], generated_texts: List[str], gt_texts: List[str]) -> Dict[str, float]:\n    metrics = {'fid': 0.0, 'diversity': 0.0, 'r_precision': 0.0, 'mm_dist': 0.0}\n    print('TODO: Implement evaluation metrics computation')\n    return metrics",
        'utils/quaternion.py': "import torch\nimport numpy as np\n_EPS4 = np.finfo(float).eps * 4.0\n_FLOAT_EPS = np.finfo(np.float64).eps\n\ndef qinv(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    mask = torch.ones_like(q)\n    mask[..., 1:] = -mask[..., 1:]\n    return q * mask\n\ndef qinv_np(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    return qinv(torch.from_numpy(q).float()).numpy()\n\ndef qnormalize(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    return q / torch.norm(q, dim=-1, keepdim=True)\n\ndef qmul(q, r):\n    assert q.shape[-1] == 4\n    assert r.shape[-1] == 4\n    original_shape = q.shape\n    terms = torch.bmm(r.view(-1, 4, 1), q.view(-1, 1, 4))\n    w = terms[:, 0, 0] - terms[:, 1, 1] - terms[:, 2, 2] - terms[:, 3, 3]\n    x = terms[:, 0, 1] + terms[:, 1, 0] - terms[:, 2, 3] + terms[:, 3, 2]\n    y = terms[:, 0, 2] + terms[:, 1, 3] + terms[:, 2, 0] - terms[:, 3, 1]\n    z = terms[:, 0, 3] - terms[:, 1, 2] + terms[:, 2, 1] + terms[:, 3, 0]\n    return torch.stack((w, x, y, z), dim=1).view(original_shape)\n\ndef qrot(q, v):\n    assert q.shape[-1] == 4\n    assert v.shape[-1] == 3\n    assert q.shape[:-1] == v.shape[:-1]\n    original_shape = list(v.shape)\n    q = q.contiguous().view(-1, 4)\n    v = v.contiguous().view(-1, 3)\n    qvec = q[:, 1:]\n    uv = torch.cross(qvec, v, dim=1)\n    uuv = torch.cross(qvec, uv, dim=1)\n    return (v + 2 * (q[:, :1] * uv + uuv)).view(original_shape)\n\ndef qeuler(q, order, epsilon=0, deg=True):\n    assert q.shape[-1] == 4\n    original_shape = list(q.shape)\n    original_shape[-1] = 3\n    q = q.view(-1, 4)\n    q0 = q[:, 0]\n    q1 = q[:, 1]\n    q2 = q[:, 2]\n    q3 = q[:, 3]\n    if order == 'xyz':\n        x = torch.atan2(2 * (q0 * q1 - q2 * q3), 1 - 2 * (q1 * q1 + q2 * q2))\n        y = torch.asin(torch.clamp(2 * (q1 * q3 + q0 * q2), -1 + epsilon, 1 - epsilon))\n        z = torch.atan2(2 * (q0 * q3 - q1 * q2), 1 - 2 * (q2 * q2 + q3 * q3))\n    elif order == 'yzx':\n        x = torch.atan2(2 * (q0 * q1 - q2 * q3), 1 - 2 * (q1 * q1 + q3 * q3))\n        y = torch.atan2(2 * (q0 * q2 - q1 * q3), 1 - 2 * (q2 * q2 + q3 * q3))\n        z = torch.asin(torch.clamp(2 * (q1 * q2 + q0 * q3), -1 + epsilon, 1 - epsilon))\n    elif order == 'zxy':\n        x = torch.asin(torch.clamp(2 * (q0 * q1 + q2 * q3), -1 + epsilon, 1 - epsilon))\n        y = torch.atan2(2 * (q0 * q2 - q1 * q3), 1 - 2 * (q1 * q1 + q2 * q2))\n        z = torch.atan2(2 * (q0 * q3 - q1 * q2), 1 - 2 * (q1 * q1 + q3 * q3))\n    elif order == 'xzy':\n        x = torch.atan2(2 * (q0 * q1 + q2 * q3), 1 - 2 * (q1 * q1 + q3 * q3))\n        y = torch.atan2(2 * (q0 * q2 + q1 * q3), 1 - 2 * (q2 * q2 + q3 * q3))\n        z = torch.asin(torch.clamp(2 * (q0 * q3 - q1 * q2), -1 + epsilon, 1 - epsilon))\n    elif order == 'yxz':\n        x = torch.asin(torch.clamp(2 * (q0 * q1 - q2 * q3), -1 + epsilon, 1 - epsilon))\n        y = torch.atan2(2 * (q1 * q3 + q0 * q2), 1 - 2 * (q1 * q1 + q2 * q2))\n        z = torch.atan2(2 * (q1 * q2 + q0 * q3), 1 - 2 * (q1 * q1 + q3 * q3))\n    elif order == 'zyx':\n        x = torch.atan2(2 * (q0 * q1 + q2 * q3), 1 - 2 * (q1 * q1 + q2 * q2))\n        y = torch.asin(torch.clamp(2 * (q0 * q2 - q1 * q3), -1 + epsilon, 1 - epsilon))\n        z = torch.atan2(2 * (q0 * q3 + q1 * q2), 1 - 2 * (q2 * q2 + q3 * q3))\n    else:\n        raise\n    if deg:\n        return torch.stack((x, y, z), dim=1).view(original_shape) * 180 / np.pi\n    else:\n        return torch.stack((x, y, z), dim=1).view(original_shape)\n\ndef qmul_np(q, r):\n    q = torch.from_numpy(q).contiguous().float()\n    r = torch.from_numpy(r).contiguous().float()\n    return qmul(q, r).numpy()\n\ndef qrot_np(q, v):\n    q = torch.from_numpy(q).contiguous().float()\n    v = torch.from_numpy(v).contiguous().float()\n    return qrot(q, v).numpy()\n\ndef qeuler_np(q, order, epsilon=0, use_gpu=False):\n    if use_gpu:\n        q = torch.from_numpy(q).cuda().float()\n        return qeuler(q, order, epsilon).cpu().numpy()\n    else:\n        q = torch.from_numpy(q).contiguous().float()\n        return qeuler(q, order, epsilon).numpy()\n\ndef qfix(q):\n    assert len(q.shape) == 3\n    assert q.shape[-1] == 4\n    result = q.copy()\n    dot_products = np.sum(q[1:] * q[:-1], axis=2)\n    mask = dot_products < 0\n    mask = (np.cumsum(mask, axis=0) % 2).astype(bool)\n    result[1:][mask] *= -1\n    return result\n\ndef euler2quat(e, order, deg=True):\n    assert e.shape[-1] == 3\n    original_shape = list(e.shape)\n    original_shape[-1] = 4\n    e = e.view(-1, 3)\n    if deg:\n        e = e * np.pi / 180.0\n    x = e[:, 0]\n    y = e[:, 1]\n    z = e[:, 2]\n    rx = torch.stack((torch.cos(x / 2), torch.sin(x / 2), torch.zeros_like(x), torch.zeros_like(x)), dim=1)\n    ry = torch.stack((torch.cos(y / 2), torch.zeros_like(y), torch.sin(y / 2), torch.zeros_like(y)), dim=1)\n    rz = torch.stack((torch.cos(z / 2), torch.zeros_like(z), torch.zeros_like(z), torch.sin(z / 2)), dim=1)\n    result = None\n    for coord in order:\n        if coord == 'x':\n            r = rx\n        elif coord == 'y':\n            r = ry\n        elif coord == 'z':\n            r = rz\n        else:\n            raise\n        if result is None:\n            result = r\n        else:\n            result = qmul(result, r)\n    if order in ['xyz', 'yzx', 'zxy']:\n        result *= -1\n    return result.view(original_shape)\n\ndef expmap_to_quaternion(e):\n    assert e.shape[-1] == 3\n    original_shape = list(e.shape)\n    original_shape[-1] = 4\n    e = e.reshape(-1, 3)\n    theta = np.linalg.norm(e, axis=1).reshape(-1, 1)\n    w = np.cos(0.5 * theta).reshape(-1, 1)\n    xyz = 0.5 * np.sinc(0.5 * theta / np.pi) * e\n    return np.concatenate((w, xyz), axis=1).reshape(original_shape)\n\ndef euler_to_quaternion(e, order):\n    assert e.shape[-1] == 3\n    original_shape = list(e.shape)\n    original_shape[-1] = 4\n    e = e.reshape(-1, 3)\n    x = e[:, 0]\n    y = e[:, 1]\n    z = e[:, 2]\n    rx = np.stack((np.cos(x / 2), np.sin(x / 2), np.zeros_like(x), np.zeros_like(x)), axis=1)\n    ry = np.stack((np.cos(y / 2), np.zeros_like(y), np.sin(y / 2), np.zeros_like(y)), axis=1)\n    rz = np.stack((np.cos(z / 2), np.zeros_like(z), np.zeros_like(z), np.sin(z / 2)), axis=1)\n    result = None\n    for coord in order:\n        if coord == 'x':\n            r = rx\n        elif coord == 'y':\n            r = ry\n        elif coord == 'z':\n            r = rz\n        else:\n            raise\n        if result is None:\n            result = r\n        else:\n            result = qmul_np(result, r)\n    if order in ['xyz', 'yzx', 'zxy']:\n        result *= -1\n    return result.reshape(original_shape)\n\ndef quaternion_to_matrix(quaternions):\n    r, i, j, k = torch.unbind(quaternions, -1)\n    two_s = 2.0 / (quaternions * quaternions).sum(-1)\n    o = torch.stack((1 - two_s * (j * j + k * k), two_s * (i * j - k * r), two_s * (i * k + j * r), two_s * (i * j + k * r), 1 - two_s * (i * i + k * k), two_s * (j * k - i * r), two_s * (i * k - j * r), two_s * (j * k + i * r), 1 - two_s * (i * i + j * j)), -1)\n    return o.reshape(quaternions.shape[:-1] + (3, 3))\n\ndef quaternion_to_matrix_np(quaternions):\n    q = torch.from_numpy(quaternions).contiguous().float()\n    return quaternion_to_matrix(q).numpy()\n\ndef quaternion_to_cont6d_np(quaternions):\n    rotation_mat = quaternion_to_matrix_np(quaternions)\n    cont_6d = np.concatenate([rotation_mat[..., 0], rotation_mat[..., 1]], axis=-1)\n    return cont_6d\n\ndef quaternion_to_cont6d(quaternions):\n    rotation_mat = quaternion_to_matrix(quaternions)\n    cont_6d = torch.cat([rotation_mat[..., 0], rotation_mat[..., 1]], dim=-1)\n    return cont_6d\n\ndef cont6d_to_matrix(cont6d):\n    assert cont6d.shape[-1] == 6, 'The last dimension must be 6'\n    x_raw = cont6d[..., 0:3]\n    y_raw = cont6d[..., 3:6]\n    x = x_raw / torch.norm(x_raw, dim=-1, keepdim=True)\n    z = torch.cross(x, y_raw, dim=-1)\n    z = z / torch.norm(z, dim=-1, keepdim=True)\n    y = torch.cross(z, x, dim=-1)\n    x = x[..., None]\n    y = y[..., None]\n    z = z[..., None]\n    mat = torch.cat([x, y, z], dim=-1)\n    return mat\n\ndef cont6d_to_matrix_np(cont6d):\n    q = torch.from_numpy(cont6d).contiguous().float()\n    return cont6d_to_matrix(q).numpy()\n\ndef matrix_to_quaternion(rotation_matrix):\n    batch_shape = rotation_matrix.shape[:-2]\n    rotation_matrix = rotation_matrix.reshape(-1, 3, 3)\n    batch_size = rotation_matrix.shape[0]\n    q = torch.zeros(batch_size, 4, device=rotation_matrix.device, dtype=rotation_matrix.dtype)\n    trace = rotation_matrix[:, 0, 0] + rotation_matrix[:, 1, 1] + rotation_matrix[:, 2, 2]\n    mask1 = trace > 0\n    s1 = torch.sqrt(trace[mask1] + 1.0) * 2\n    q[mask1, 0] = 0.25 * s1\n    q[mask1, 1] = (rotation_matrix[mask1, 2, 1] - rotation_matrix[mask1, 1, 2]) / s1\n    q[mask1, 2] = (rotation_matrix[mask1, 0, 2] - rotation_matrix[mask1, 2, 0]) / s1\n    q[mask1, 3] = (rotation_matrix[mask1, 1, 0] - rotation_matrix[mask1, 0, 1]) / s1\n    mask2 = ~mask1 & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 1, 1]) & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 2, 2])\n    s2 = torch.sqrt(1.0 + rotation_matrix[mask2, 0, 0] - rotation_matrix[mask2, 1, 1] - rotation_matrix[mask2, 2, 2]) * 2\n    q[mask2, 0] = (rotation_matrix[mask2, 2, 1] - rotation_matrix[mask2, 1, 2]) / s2\n    q[mask2, 1] = 0.25 * s2\n    q[mask2, 2] = (rotation_matrix[mask2, 0, 1] + rotation_matrix[mask2, 1, 0]) / s2\n    q[mask2, 3] = (rotation_matrix[mask2, 0, 2] + rotation_matrix[mask2, 2, 0]) / s2\n    mask3 = ~mask1 & ~mask2 & (rotation_matrix[:, 1, 1] > rotation_matrix[:, 2, 2])\n    s3 = torch.sqrt(1.0 + rotation_matrix[mask3, 1, 1] - rotation_matrix[mask3, 0, 0] - rotation_matrix[mask3, 2, 2]) * 2\n    q[mask3, 0] = (rotation_matrix[mask3, 0, 2] - rotation_matrix[mask3, 2, 0]) / s3\n    q[mask3, 1] = (rotation_matrix[mask3, 0, 1] + rotation_matrix[mask3, 1, 0]) / s3\n    q[mask3, 2] = 0.25 * s3\n    q[mask3, 3] = (rotation_matrix[mask3, 1, 2] + rotation_matrix[mask3, 2, 1]) / s3\n    mask4 = ~mask1 & ~mask2 & ~mask3\n    s4 = torch.sqrt(1.0 + rotation_matrix[mask4, 2, 2] - rotation_matrix[mask4, 0, 0] - rotation_matrix[mask4, 1, 1]) * 2\n    q[mask4, 0] = (rotation_matrix[mask4, 1, 0] - rotation_matrix[mask4, 0, 1]) / s4\n    q[mask4, 1] = (rotation_matrix[mask4, 0, 2] + rotation_matrix[mask4, 2, 0]) / s4\n    q[mask4, 2] = (rotation_matrix[mask4, 1, 2] + rotation_matrix[mask4, 2, 1]) / s4\n    q[mask4, 3] = 0.25 * s4\n    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)\n    return q.reshape(batch_shape + (4,))\n\ndef matrix_to_quaternion_np(rotation_matrix):\n    mat = torch.from_numpy(rotation_matrix).contiguous().float()\n    return matrix_to_quaternion(mat).numpy()\n\ndef cont6d_to_quaternion(cont6d):\n    mat = cont6d_to_matrix(cont6d)\n    return matrix_to_quaternion(mat)\n\ndef cont6d_to_quaternion_np(cont6d):\n    q = torch.from_numpy(cont6d).contiguous().float()\n    return cont6d_to_quaternion(q).numpy()\n\ndef qpow(q0, t, dtype=torch.float):\n    q0 = qnormalize(q0)\n    theta0 = torch.acos(q0[..., 0])\n    mask = (theta0 <= 1e-09) * (theta0 >= -1e-09)\n    theta0 = (1 - mask) * theta0 + mask * 1e-09\n    v0 = q0[..., 1:] / torch.sin(theta0).view(-1, 1)\n    if isinstance(t, torch.Tensor):\n        q = torch.zeros(t.shape + q0.shape)\n        theta = t.view(-1, 1) * theta0.view(1, -1)\n    else:\n        q = torch.zeros(q0.shape)\n        theta = t * theta0\n    q[..., 0] = torch.cos(theta)\n    q[..., 1:] = v0 * torch.sin(theta).unsqueeze(-1)\n    return q.to(dtype)\n\ndef qslerp(q0, q1, t):\n    q0 = qnormalize(q0)\n    q1 = qnormalize(q1)\n    q_ = qpow(qmul(q1, qinv(q0)), t)\n    return qmul(q_, q0.contiguous().view(torch.Size([1] * len(t.shape)) + q0.shape).expand(t.shape + q0.shape).contiguous())\n\ndef qbetween(v0, v1):\n    assert v0.shape[-1] == 3, 'v0 must be of the shape (*, 3)'\n    assert v1.shape[-1] == 3, 'v1 must be of the shape (*, 3)'\n    v = torch.cross(v0, v1)\n    w = torch.sqrt((v0 ** 2).sum(dim=-1, keepdim=True) * (v1 ** 2).sum(dim=-1, keepdim=True)) + (v0 * v1).sum(dim=-1, keepdim=True)\n    return qnormalize(torch.cat([w, v], dim=-1))\n\ndef qbetween_np(v0, v1):\n    assert v0.shape[-1] == 3, 'v0 must be of the shape (*, 3)'\n    assert v1.shape[-1] == 3, 'v1 must be of the shape (*, 3)'\n    v0 = torch.from_numpy(v0).float()\n    v1 = torch.from_numpy(v1).float()\n    return qbetween(v0, v1).numpy()\n\ndef lerp(p0, p1, t):\n    if not isinstance(t, torch.Tensor):\n        t = torch.Tensor([t])\n    new_shape = t.shape + p0.shape\n    new_view_t = t.shape + torch.Size([1] * len(p0.shape))\n    new_view_p = torch.Size([1] * len(t.shape)) + p0.shape\n    p0 = p0.view(new_view_p).expand(new_shape)\n    p1 = p1.view(new_view_p).expand(new_shape)\n    t = t.view(new_view_t).expand(new_shape)\n    return p0 + t * (p1 - p0)",
        'utils/skeleton.py': "import numpy as np\nimport torch\nfrom .quaternion import *\nimport scipy.ndimage.filters as filters\n\nclass Skeleton(object):\n\n    def __init__(self, offset, kinematic_tree, device):\n        self.device = device\n        if isinstance(offset, np.ndarray):\n            self._raw_offset_np = offset.copy()\n            self._raw_offset = torch.from_numpy(offset).to(device).float()\n        else:\n            self._raw_offset_np = offset.detach().cpu().numpy()\n            self._raw_offset = offset.clone().detach().to(device).float()\n        self._kinematic_tree = kinematic_tree\n        self._offset = None\n        self._parents = [0] * len(self._raw_offset)\n        self._parents[0] = -1\n        for chain in self._kinematic_tree:\n            for j in range(1, len(chain)):\n                self._parents[chain[j]] = chain[j - 1]\n\n    def njoints(self):\n        return len(self._raw_offset)\n\n    def offset(self):\n        return self._offset\n\n    def set_offset(self, offsets):\n        self._offset = offsets.clone().detach().to(self.device).float()\n\n    def kinematic_tree(self):\n        return self._kinematic_tree\n\n    def parents(self):\n        return self._parents\n\n    def get_offsets_joints_batch(self, joints):\n        assert len(joints.shape) == 3\n        _offsets = self._raw_offset.expand(joints.shape[0], -1, -1).clone()\n        for i in range(1, self._raw_offset.shape[0]):\n            _offsets[:, i] = torch.norm(joints[:, i] - joints[:, self._parents[i]], p=2, dim=1)[:, None] * _offsets[:, i]\n        self._offset = _offsets.detach()\n        return _offsets\n\n    def get_offsets_joints(self, joints):\n        assert len(joints.shape) == 2\n        _offsets = self._raw_offset.clone()\n        for i in range(1, self._raw_offset.shape[0]):\n            _offsets[i] = torch.norm(joints[i] - joints[self._parents[i]], p=2, dim=0) * _offsets[i]\n        self._offset = _offsets.detach()\n        return _offsets\n\n    def inverse_kinematics_np(self, joints, face_joint_idx, smooth_forward=False):\n        assert len(face_joint_idx) == 4\n        'Get Forward Direction'\n        l_hip, r_hip, sdr_r, sdr_l = face_joint_idx\n        across1 = joints[:, r_hip] - joints[:, l_hip]\n        across2 = joints[:, sdr_r] - joints[:, sdr_l]\n        across = across1 + across2\n        across = across / np.sqrt((across ** 2).sum(axis=-1))[:, np.newaxis]\n        forward = np.cross(np.array([[0, 1, 0]]), across, axis=-1)\n        if smooth_forward:\n            forward = filters.gaussian_filter1d(forward, 20, axis=0, mode='nearest')\n        forward = forward / np.sqrt((forward ** 2).sum(axis=-1))[..., np.newaxis]\n        'Get Root Rotation'\n        target = np.array([[0, 0, 1]]).repeat(len(forward), axis=0)\n        root_quat = qbetween_np(forward, target)\n        'Inverse Kinematics'\n        quat_params = np.zeros(joints.shape[:-1] + (4,))\n        root_quat[0] = np.array([[1.0, 0.0, 0.0, 0.0]])\n        quat_params[:, 0] = root_quat\n        for chain in self._kinematic_tree:\n            R = root_quat\n            for j in range(len(chain) - 1):\n                u = self._raw_offset_np[chain[j + 1]][np.newaxis, ...].repeat(len(joints), axis=0)\n                v = joints[:, chain[j + 1]] - joints[:, chain[j]]\n                v = v / np.sqrt((v ** 2).sum(axis=-1))[:, np.newaxis]\n                rot_u_v = qbetween_np(u, v)\n                R_loc = qmul_np(qinv_np(R), rot_u_v)\n                quat_params[:, chain[j + 1], :] = R_loc\n                R = qmul_np(R, R_loc)\n        return quat_params\n\n    def forward_kinematics(self, quat_params, root_pos, skel_joints=None, do_root_R=True):\n        if skel_joints is not None:\n            offsets = self.get_offsets_joints_batch(skel_joints)\n        if len(self._offset.shape) == 2:\n            offsets = self._offset.expand(quat_params.shape[0], -1, -1)\n        joints = torch.zeros(quat_params.shape[:-1] + (3,)).to(self.device)\n        joints[:, 0] = root_pos\n        for chain in self._kinematic_tree:\n            if do_root_R:\n                R = quat_params[:, 0]\n            else:\n                R = torch.tensor([[1.0, 0.0, 0.0, 0.0]]).expand(len(quat_params), -1).detach().to(self.device)\n            for i in range(1, len(chain)):\n                R = qmul(R, quat_params[:, chain[i]])\n                offset_vec = offsets[:, chain[i]]\n                joints[:, chain[i]] = qrot(R, offset_vec) + joints[:, chain[i - 1]]\n        return joints\n\n    def forward_kinematics_np(self, quat_params, root_pos, skel_joints=None, do_root_R=True):\n        if skel_joints is not None:\n            skel_joints = torch.from_numpy(skel_joints)\n            offsets = self.get_offsets_joints_batch(skel_joints)\n        if len(self._offset.shape) == 2:\n            offsets = self._offset.expand(quat_params.shape[0], -1, -1)\n        offsets = offsets.numpy()\n        joints = np.zeros(quat_params.shape[:-1] + (3,))\n        joints[:, 0] = root_pos\n        for chain in self._kinematic_tree:\n            if do_root_R:\n                R = quat_params[:, 0]\n            else:\n                R = np.array([[1.0, 0.0, 0.0, 0.0]]).repeat(len(quat_params), axis=0)\n            for i in range(1, len(chain)):\n                R = qmul_np(R, quat_params[:, chain[i]])\n                offset_vec = offsets[:, chain[i]]\n                joints[:, chain[i]] = qrot_np(R, offset_vec) + joints[:, chain[i - 1]]\n        return joints\n\n    def forward_kinematics_cont6d_np(self, cont6d_params, root_pos, skel_joints=None, do_root_R=True):\n        if skel_joints is not None:\n            skel_joints = torch.from_numpy(skel_joints)\n            offsets = self.get_offsets_joints_batch(skel_joints)\n        if len(self._offset.shape) == 2:\n            offsets = self._offset.expand(cont6d_params.shape[0], -1, -1)\n        offsets = offsets.numpy()\n        joints = np.zeros(cont6d_params.shape[:-1] + (3,))\n        joints[:, 0] = root_pos\n        for chain in self._kinematic_tree:\n            if do_root_R:\n                matR = cont6d_to_matrix_np(cont6d_params[:, 0])\n            else:\n                matR = np.eye(3)[np.newaxis, :].repeat(len(cont6d_params), axis=0)\n            for i in range(1, len(chain)):\n                matR = np.matmul(matR, cont6d_to_matrix_np(cont6d_params[:, chain[i]]))\n                offset_vec = offsets[:, chain[i]][..., np.newaxis]\n                joints[:, chain[i]] = np.matmul(matR, offset_vec).squeeze(-1) + joints[:, chain[i - 1]]\n        return joints\n\n    def forward_kinematics_cont6d(self, cont6d_params, root_pos, skel_joints=None, do_root_R=True):\n        if skel_joints is not None:\n            offsets = self.get_offsets_joints_batch(skel_joints)\n        if len(self._offset.shape) == 2:\n            offsets = self._offset.expand(cont6d_params.shape[0], -1, -1)\n        joints = torch.zeros(cont6d_params.shape[:-1] + (3,)).to(cont6d_params.device)\n        joints[..., 0, :] = root_pos\n        for chain in self._kinematic_tree:\n            if do_root_R:\n                matR = cont6d_to_matrix(cont6d_params[:, 0])\n            else:\n                matR = torch.eye(3).expand((len(cont6d_params), -1, -1)).detach().to(cont6d_params.device)\n            for i in range(1, len(chain)):\n                matR = torch.matmul(matR, cont6d_to_matrix(cont6d_params[:, chain[i]]))\n                offset_vec = offsets[:, chain[i]].unsqueeze(-1)\n                joints[:, chain[i]] = torch.matmul(matR, offset_vec).squeeze(-1) + joints[:, chain[i - 1]]\n        return joints",
        'utils/train_utils.py': "import math\nimport copy\nimport os\nimport time\nimport torch\nimport torch.nn.functional as F\nfrom torch.cuda.amp import autocast, GradScaler\nfrom torch.utils.data import DataLoader\nfrom typing import Optional, List, Union, Tuple\nfrom tqdm import tqdm\nfrom utils.wandb_logger import WandbLogger\n\ndef build_prev_and_clean_diffs(hist: torch.Tensor, future: torch.Tensor, joint_count: int=22) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n    B, T_hist, _ = hist.shape\n    last_frame = hist[:, -1]\n    target_frame = future[:, 0]\n\n    def get_pos(frame):\n        ric_21 = frame[:, 4:67].reshape(B, 21, 3)\n        root_pos = torch.zeros((B, 1, 3), device=frame.device, dtype=frame.dtype)\n        return torch.cat([root_pos, ric_21], dim=1)\n    prev_pos = get_pos(last_frame)\n\n    def get_rot(frame):\n        rot_21 = frame[:, 67:193].reshape(B, 21, 6)\n        root_rot = torch.zeros((B, 1, 6), device=frame.device, dtype=frame.dtype)\n        root_rot[:, 0, 0] = 1.0\n        root_rot[:, 0, 4] = 1.0\n        return torch.cat([root_rot, rot_21], dim=1)\n    prev_rot6d = get_rot(last_frame)\n    prev_v = last_frame[:, 193:259].contiguous().view(B, 22, 3)\n    clean_v = target_frame[:, 193:259].contiguous().view(B, 22, 3)\n    return (prev_pos, prev_rot6d, prev_v, clean_v)\n\ndef train(motion_history_encoder: torch.nn.Module, flow_predictor: torch.nn.Module, dataloader: DataLoader, num_epochs: int, save_dir: str, device: str='cuda', lr: float=0.0001, weight_decay: float=0.01, max_grad_norm: float=1.0, ema_decay: float=0.9999, wandb_project: Optional[str]=None, wandb_run_name: Optional[str]=None):\n    os.makedirs(save_dir, exist_ok=True)\n    motion_history_encoder.to(device)\n    flow_predictor.to(device)\n    wandb_logger = None\n    if wandb_project:\n        config = {'lr': lr, 'weight_decay': weight_decay, 'max_grad_norm': max_grad_norm, 'ema_decay': ema_decay, 'num_epochs': num_epochs, 'batch_size': dataloader.batch_size, 'mhe_params': sum((p.numel() for p in motion_history_encoder.parameters())), 'fp_params': sum((p.numel() for p in flow_predictor.parameters()))}\n        wandb_logger = WandbLogger(project=wandb_project, name=wandb_run_name, config=config)\n\n    def copy_model(m):\n        ema = copy.deepcopy(m)\n        for p in ema.parameters():\n            p.requires_grad_(False)\n        return ema\n    ema_mhe = copy_model(motion_history_encoder)\n    ema_fmp = copy_model(flow_predictor)\n    params = list(motion_history_encoder.parameters()) + list(flow_predictor.parameters())\n    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)\n    scaler = GradScaler()\n    total_steps = num_epochs * len(dataloader)\n    print(f'Training for {num_epochs} epochs, total {total_steps} steps.')\n\n    def lr_lambda(step):\n        warmup = max(1, int(0.02 * total_steps))\n        if step < warmup:\n            return float(step + 1) / float(warmup)\n        progress = (step - warmup) / float(max(1, total_steps - warmup))\n        return 0.5 * (1.0 + math.cos(math.pi * progress))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n    global_step = 0\n    best_loss = float('inf')\n    best_epoch = -1\n    model_log_interval = 10\n    motion_history_encoder.train()\n    flow_predictor.train()\n\n    def save_checkpoint(filename: str, loss: float):\n        path = os.path.join(save_dir, filename)\n        torch.save({'motion_history_encoder': motion_history_encoder.state_dict(), 'flow_predictor': flow_predictor.state_dict(), 'ema_mhe': ema_mhe.state_dict(), 'ema_fmp': ema_fmp.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(), 'scheduler': scheduler.state_dict(), 'epoch': epoch, 'global_step': global_step, 'loss': loss}, path)\n        print(f'Saved checkpoint: {path}')\n    try:\n        for epoch in tqdm(range(num_epochs), desc='Training', unit='epoch'):\n            epoch_loss = 0.0\n            num_batches = 0\n            pbar = tqdm(dataloader, desc=f'Epoch {epoch}', leave=False, unit='batch')\n            batch_start_time = time.time()\n            for batch in pbar:\n                motion = batch['motion'].to(device)\n                B, T, _ = motion.shape\n                text_embeddings = batch.get('text_clip', batch.get('captions'))\n                if hasattr(text_embeddings, 'to'):\n                    text_embeddings = text_embeddings.to(device)\n                if 'duration' in batch:\n                    duration = batch['duration'].to(device)\n                else:\n                    lengths = batch['lengths'].to(device).float()\n                    duration = (lengths / 200.0).unsqueeze(-1)\n                global_dropout_prob = 0.05\n                cond_dropout_prob = 0.1\n                is_global_uncond = torch.rand(1) < global_dropout_prob\n                is_zero_shot = torch.rand(1) < 0.1 and T > 0\n                t_prog = None\n                prev_features = None\n                clean_diffs = None\n                text_input = None\n                dur_input = None\n                hist_input_for_encoder = None\n                future = None\n                if is_zero_shot:\n                    future = motion[:, 0:1]\n                    text_input = None if is_global_uncond or torch.rand(1) < cond_dropout_prob else text_embeddings\n                    dur_input = None if is_global_uncond or torch.rand(1) < cond_dropout_prob else duration\n                    hist_input_for_encoder = None\n                    t_prog = None\n                    prev_features = None\n                    clean_diffs = future[:, 0, 193:259].contiguous().view(B, 22, 3)\n                else:\n                    idx_limit = min(int(batch['lengths'].min().item()) if 'lengths' in batch else T, T)\n                    end_idx = torch.randint(1, idx_limit - 1, (1,)).item()\n                    start_idx = torch.randint(0, end_idx, (1,)).item()\n                    hist_actual_slice = motion[:, start_idx:end_idx]\n                    future = motion[:, start_idx:end_idx + 1]\n                    text_input = None if is_global_uncond or torch.rand(1) < cond_dropout_prob else text_embeddings\n                    dur_input = None if is_global_uncond or torch.rand(1) < cond_dropout_prob else duration\n                    hist_input_for_encoder = None if is_global_uncond or torch.rand(1) < cond_dropout_prob else hist_actual_slice\n                    prev_pos, prev_rot6d, prev_v, clean_diffs = build_prev_and_clean_diffs(hist_actual_slice, future)\n                    prev_features = torch.cat([prev_pos, prev_rot6d, prev_v], dim=-1)\n                    prog = float(end_idx) / float(T)\n                    t_prog = torch.full((B,), prog, device=device, dtype=torch.float32)\n                assert future is not None, 'future not initialized!'\n                assert clean_diffs is not None, 'clean_diffs not initialized!'\n                assert text_input is None or text_input.shape[0] == B, f'text_input batch size mismatch: {text_input.shape[0]} != {B}'\n                assert dur_input is None or dur_input.shape[0] == B, f'dur_input batch size mismatch: {dur_input.shape[0]} != {B}'\n                assert t_prog is None or t_prog.shape[0] == B, f't_prog batch size mismatch: {t_prog.shape[0]} != {B}'\n                optimizer.zero_grad(set_to_none=True)\n                with autocast(dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):\n                    history_context = motion_history_encoder(text=text_input, input_features=hist_input_for_encoder, batch_size=B)\n                    eps = torch.randn_like(clean_diffs)\n                    t = torch.rand(B, device=device)\n                    t_b = t.view(B, 1, 1)\n                    noisy_target_diffs = t_b * clean_diffs + (1.0 - t_b) * eps\n                    pred_eps = flow_predictor(history_features=history_context, noise_level=t, noisy_target=noisy_target_diffs, prev_frame_features=prev_features)\n                    flow_target = clean_diffs - eps\n                    loss = F.mse_loss(pred_eps, flow_target)\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                grad_norm = torch.nn.utils.clip_grad_norm_(params, max_grad_norm)\n                scaler.step(optimizer)\n                scaler.update()\n                scheduler.step()\n                with torch.no_grad():\n                    for ema_p, p in zip(ema_mhe.parameters(), motion_history_encoder.parameters()):\n                        ema_p.data.mul_(ema_decay).add_(p.data, alpha=1 - ema_decay)\n                    for ema_p, p in zip(ema_fmp.parameters(), flow_predictor.parameters()):\n                        ema_p.data.mul_(ema_decay).add_(p.data, alpha=1 - ema_decay)\n                batch_time = time.time() - batch_start_time\n                batch_start_time = time.time()\n                gpu_memory_allocated = 0.0\n                gpu_memory_reserved = 0.0\n                if torch.cuda.is_available():\n                    gpu_memory_allocated = torch.cuda.memory_allocated() / 1000000000.0\n                    gpu_memory_reserved = torch.cuda.memory_reserved() / 1000000000.0\n                pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})\n                if wandb_logger is not None:\n                    wandb_logger.log({'train/loss': loss.item(), 'train/lr': scheduler.get_last_lr()[0], 'train/epoch': epoch, 'train/grad_norm': grad_norm.item() if hasattr(grad_norm, 'item') else grad_norm, 'train/batch_time': batch_time, 'train/samples_per_sec': B / batch_time if batch_time > 0 else 0, 'system/gpu_memory_allocated_gb': gpu_memory_allocated, 'system/gpu_memory_reserved_gb': gpu_memory_reserved}, step=global_step)\n                if global_step % 100 == 0:\n                    current_lr = scheduler.get_last_lr()[0]\n                    tqdm.write(f'[Epoch {epoch}] [Step {global_step}] loss={loss.item():.6f} lr={current_lr:.2e}')\n                epoch_loss += loss.item()\n                num_batches += 1\n                global_step += 1\n            pbar.close()\n            avg_epoch_loss = epoch_loss / max(1, num_batches)\n            tqdm.write(f'==> End of Epoch {epoch}: Avg Loss = {avg_epoch_loss:.6f}')\n            if wandb_logger is not None:\n                wandb_logger.log({'epoch/avg_loss': avg_epoch_loss, 'epoch/num': epoch}, step=global_step)\n            save_checkpoint('latest.pt', avg_epoch_loss)\n            if avg_epoch_loss < best_loss:\n                tqdm.write(f'New best model! (Loss: {best_loss:.6f} -> {avg_epoch_loss:.6f})')\n                best_loss = avg_epoch_loss\n                save_checkpoint('best.pt', avg_epoch_loss)\n                if wandb_logger is not None and epoch - best_epoch >= model_log_interval:\n                    wandb_logger.log_model(os.path.join(save_dir, 'best.pt'), f'best-model-epoch-{epoch}', description=f'Best model at epoch {epoch} with loss {avg_epoch_loss:.6f}')\n                    best_epoch = epoch\n    except KeyboardInterrupt:\n        tqdm.write('Training interrupted. Saving emergency checkpoint...')\n        save_checkpoint('latest_interrupted.pt', 0.0)\n        tqdm.write('Done.')\n    if wandb_logger is not None:\n        wandb_logger.log_summary({'best_loss': best_loss})\n        wandb_logger.finish()\n    return (ema_mhe, ema_fmp)",
        'utils/text_encoder.py': '"""\nText encoding utility using CLIP model from Hugging Face Transformers.\n"""\n\nimport torch\nfrom transformers import CLIPTokenizer, CLIPTextModel\nfrom typing import List, Union\n\n\nclass CLIPEncoder(torch.nn.Module):\n    """\n    Utility class to encode text captions using Microsoft\'s CLIP model.\n    By default, uses \'openai/clip-vit-base-patch32\' which produces 512D embeddings.\n    """\n\n    def __init__(\n        self,\n        model_name: str = "openai/clip-vit-base-patch32",\n    ):\n        super().__init__()\n\n        print(f"Loading CLIP model \'{model_name}\'...")\n        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)\n        self.model = CLIPTextModel.from_pretrained(model_name)\n        self.model.eval()\n\n        # Freeze CLIP parameters\n        for param in self.model.parameters():\n            param.requires_grad = False\n\n    @torch.no_grad()\n    def forward(self, text: Union[str, List[str]]) -> torch.Tensor:\n        """\n        Encode a list of captions or a single caption into embeddings.\n\n        Args:\n            text: A single string or a list of strings.\n\n        Returns:\n            embeddings: (B, 512) tensor containing the text embeddings.\n        """\n        if isinstance(text, str):\n            text = [text]\n\n        # Determine device dynamically\n        device = next(self.model.parameters()).device\n\n        inputs = self.tokenizer(\n            text, padding=True, truncation=True, return_tensors="pt"\n        ).to(device)\n        outputs = self.model(**inputs)\n\n        # Use the pooler_output for a global representation of the sentence\n        # Shape: (Batch_Size, 512)\n        embeddings = outputs.pooler_output\n\n        return embeddings\n\n    @property\n    def embedding_dim(self) -> int:\n        """Output dimension of the CLIP text model."""\n        return self.model.config.hidden_size\n',
        'utils/wandb_logger.py': "import os\nimport sys\nfrom datetime import datetime\nfrom typing import Optional, Dict, Any\ntry:\n    import wandb\n    WANDB_AVAILABLE = True\nexcept ImportError:\n    WANDB_AVAILABLE = False\n    wandb = None\n\ndef is_kaggle_environment() -> bool:\n    return os.path.exists('/kaggle') or 'kaggle' in sys.executable.lower()\n\ndef get_kaggle_secret(secret_name: str) -> Optional[str]:\n    if not is_kaggle_environment():\n        return None\n    try:\n        from kaggle_secrets import UserSecretsClient\n        user_secrets = UserSecretsClient()\n        return user_secrets.get_secret(secret_name)\n    except Exception:\n        return None\n\nclass WandbLogger:\n\n    def __init__(self, project: str, name: Optional[str]=None, config: Optional[Dict[str, Any]]=None, kaggle_secret_name: str='WANDB_API_KEY', enabled: bool=True):\n        self.project = project\n        self.config = config or {}\n        self.enabled = enabled and WANDB_AVAILABLE\n        self.run = None\n        if name is None:\n            self.name = 'motion-generation-buet'\n        else:\n            self.name = name\n        if not self.enabled:\n            if not WANDB_AVAILABLE:\n                print('[WandbLogger] wandb not installed. Logging disabled.')\n            elif not enabled:\n                print('[WandbLogger] Logging disabled by user.')\n            return\n        self._authenticate(kaggle_secret_name)\n        try:\n            self.run = wandb.init(project=project, entity='motion-generation-buet', config=config, reinit=True)\n            print(f'[WandbLogger] Initialized run: {self.run.name}')\n            print(f'[WandbLogger] View at: {self.run.url}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to initialize: {e}')\n            self.enabled = False\n\n    def _authenticate(self, secret_name: str) -> None:\n        api_key = os.environ.get('WANDB_API_KEY')\n        if api_key is None:\n            api_key = get_kaggle_secret(secret_name)\n        if api_key:\n            try:\n                wandb.login(key=api_key)\n                print('[WandbLogger] Authenticated successfully.')\n            except Exception as e:\n                print(f'[WandbLogger] Authentication failed: {e}')\n        else:\n            print('[WandbLogger] No API key found. Using existing login or anonymous mode.')\n\n    def log(self, metrics: Dict[str, Any], step: Optional[int]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            wandb.log(metrics, step=step)\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log metrics: {e}')\n\n    def log_model(self, path: str, name: str, description: Optional[str]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            artifact = wandb.Artifact(name, type='model', description=description)\n            artifact.add_file(path)\n            self.run.log_artifact(artifact)\n            print(f'[WandbLogger] Logged model artifact: {name}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log model: {e}')\n\n    def log_summary(self, metrics: Dict[str, Any]) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            for key, value in metrics.items():\n                wandb.run.summary[key] = value\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log summary: {e}')\n\n    def finish(self) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            wandb.finish()\n            print('[WandbLogger] Run finished.')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to finish run: {e}')\n\n    def __enter__(self) -> 'WandbLogger':\n        return self\n\n    def __exit__(self, exc_type, exc_val, exc_tb) -> None:\n        self.finish()",
    }
    
    for filepath, content in FILES.items():
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f'Created {filepath}')
    
    # Install dependencies
    print("Installing dependencies (this may take a minute)...")
    %pip install -r requirements.txt
    
    # Copy dataset
    print("Copying dataset...")
    !apt -qq install rclone && rclone copy /kaggle/input/ /kaggle/working/dataset/ --transfers 16 --checkers 16 --progress --ignore-existing -q
    
    print("Setup Complete!")
else:
    print("Running locally. No setup needed.")


# Project Setup for Colab and Kaggle

This notebook was automatically bundled for cloud execution. Run the cell below to reconstruct the project structure and install dependencies.

# Stage 3 (Updated): Autoregressive Flow Matching Motion Generation

This notebook keeps the Stage 3 base structure and updates the modeling/training/inference pipeline with:
- GRU context encoder unchanged
- Deeper FlowNet replacement
- Curriculum horizon training + scheduled sampling
- EMA checkpoints
- CFG-style conditional inference

In [ ]:
# Imports
import copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from IPython.display import display

from config import Config
from models import KinematicChainEncoder
from utils.dataset import Text2MotionDataset, text2motion_collate_fn
from utils.motion_utils import (
    T2M_KINEMATIC_CHAIN as t2m_kinematic_chain,
    T2M_KINEMATIC_CHAIN,
    T2M_RAW_OFFSETS,
    _forward_kinematics,
    features_to_positions,
)
from utils.quaternion import cont6d_to_quaternion, qrot
from utils.text_encoder import CLIPEncoder
from utils.visualization import visualize_motion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

In [ ]:
# Config + data
config = Config()
config.dataset_path = Path("./dataset/humanml3d-subset")

mean = np.load(config.dataset_path / "Mean.npy")
std = np.load(config.dataset_path / "Std.npy")
config.motion_dim = int(mean.shape[0])
print(f"Using motion dim: {config.motion_dim}")

train_dataset = Text2MotionDataset(config, mean, std, split="train")
val_dataset = Text2MotionDataset(config, mean, std, split="val")

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=device.type == "cuda",
    drop_last=False,
    collate_fn=text2motion_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=device.type == "cuda",
    drop_last=False,
    collate_fn=text2motion_collate_fn,
)

mean_t = torch.from_numpy(mean).float().to(device)
std_t = torch.from_numpy(std).float().to(device)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# Text encoder (CLIP, frozen)
text_encoder = CLIPEncoder().to(device)
for p in text_encoder.parameters():
    p.requires_grad = False


def encode_text(captions):
    with torch.no_grad():
        emb = text_encoder(captions)
    return emb.to(device)

In [ ]:
class TextConditionedGRUContext(nn.Module):
    """GRU context encoder for flow matching."""

    def __init__(
        self,
        motion_dim: int,
        text_dim: int,
        hidden_dim: int,
        num_layers: int = 1,
        text_scale: float = 1.0,
    ):
        super().__init__()
        self.motion_dim = motion_dim
        self.text_dim = text_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.text_scale = text_scale
        self.text_to_hidden = nn.Linear(text_dim, hidden_dim)
        self.gru = nn.GRU(
            input_size=motion_dim + text_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

    def init_hidden(self, text_emb):
        h0 = self.text_to_hidden(text_emb)
        h0 = h0.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return h0

    def step(self, x_t, text_emb, h):
        text_step = self.text_scale * text_emb
        gru_input = torch.cat([x_t, text_step], dim=-1).unsqueeze(1)
        h_seq, h = self.gru(gru_input, h)
        h_t = h_seq[:, 0, :]
        return h_t, h


class FlowNet(nn.Module):
    """
    Spatial Transformer Flow Matching on rotation tokens.
    Input:  h_t (B,512), x_noisy (B,135), t (B,)
    Output: velocity (B,135) in rotation space
    135D = root_pos(3) + 22*rot6d(132)
    Bone lengths fixed by FK at inference and never predicted directly.
    """

    def __init__(
        self,
        context_dim=512,
        model_dim=256,
        num_layers=4,
        num_heads=4,
        time_embed_dim=64,
        dropout=0.1,
    ):
        super().__init__()
        self.time_embed_dim = time_embed_dim

        self.root_proj = nn.Linear(3, model_dim)
        self.joint_proj = nn.Linear(6, model_dim)

        self.kinematic_encoder = KinematicChainEncoder(model_dim)

        self.gru_proj = nn.Linear(context_dim, model_dim)

        self.time_mlp = nn.Sequential(
            nn.Linear(time_embed_dim, model_dim),
            nn.SiLU(),
            nn.Linear(model_dim, model_dim),
        )

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=model_dim,
                nhead=num_heads,
                dim_feedforward=model_dim * 4,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            ),
            num_layers=num_layers,
            enable_nested_tensor=False,
        )

        self.root_head = nn.Sequential(
            nn.Linear(model_dim, model_dim),
            nn.GELU(),
            nn.Linear(model_dim, 3),
        )
        self.joint_head = nn.Sequential(
            nn.Linear(model_dim, model_dim),
            nn.GELU(),
            nn.Linear(model_dim, 6),
        )

    def _time_emb(self, t):
        half = self.time_embed_dim // 2
        freqs = torch.exp(
            -torch.log(torch.tensor(10000.0, device=t.device))
            / (half - 1)
            * torch.arange(half, device=t.device)
        )
        args = t.unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    def forward(self, h_t, x_noisy, t):
        B = h_t.shape[0]

        root_tok = self.root_proj(x_noisy[:, :3].unsqueeze(1))
        joint_tok = self.joint_proj(x_noisy[:, 3:].reshape(B, 22, 6))

        joint_ids = torch.arange(22, device=h_t.device)
        joint_tok = joint_tok + self.kinematic_encoder(joint_ids).unsqueeze(0)

        tokens = torch.cat([root_tok, joint_tok], dim=1)

        tokens = tokens + self.gru_proj(h_t).unsqueeze(1)

        tokens = tokens + self.time_mlp(self._time_emb(t)).unsqueeze(1)

        tokens = self.transformer(tokens)

        root_vel = self.root_head(tokens[:, 0])
        joint_vel = self.joint_head(tokens[:, 1:])
        return torch.cat([root_vel, joint_vel.reshape(B, 132)], dim=-1)


class EMAModel:
    def __init__(self, model, decay=0.999):
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k], alpha=1.0 - self.decay)
            else:
                v.copy_(msd[k])

In [ ]:
# Models — keep GRU exactly, replace FlowNet
gru = TextConditionedGRUContext(
    motion_dim=config.motion_dim,
    text_dim=512,
    hidden_dim=512,
    num_layers=2,
    text_scale=0.3,
).to(device)

flow = FlowNet(
    context_dim=512,
    model_dim=256,
    num_layers=4,
    num_heads=4,
    time_embed_dim=64,
    dropout=0.1,
).to(device)

In [ ]:
# hyperparameters
num_epochs = 800
run_training = True  # Set True only when you want to continue training

cfg_dropout = 0.1
curriculum_start = 20
curriculum_step = 10
curriculum_step_epochs = 50
max_horizon = 40
ss_start_epoch = 150
ss_warmup_epochs = 100
ss_max_prob = 0.15
ss_rollout_steps = 10

current_horizon = curriculum_start
mean = np.load("./dataset/humanml3d-subset/Mean.npy")
std = np.load("./dataset/humanml3d-subset/Std.npy")
mean_t = torch.from_numpy(mean).float().to(device)
std_t = torch.from_numpy(std).float().to(device)


class Normalizer:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def normalize(self, x):
        return (x - self.mean) / self.std

    def denormalize(self, x):
        return x * self.std + self.mean


normalizer = Normalizer(mean_t, std_t)

optimizer = torch.optim.AdamW(
    list(gru.parameters()) + list(flow.parameters()),
    lr=1e-4,
    weight_decay=1e-5,
)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
gru_ema = EMAModel(gru, decay=0.999)
flow_ema = EMAModel(flow, decay=0.999)


def root_features_to_root_positions(
    root_features: torch.Tensor,
    prev_root_pos: torch.Tensor,
) -> torch.Tensor:
    """
    Convert velocity-form root features into absolute root positions.

    The velocity-form layout is `[root_height_y, root_vel_x, root_vel_z]`.
    `prev_root_pos` provides the absolute XYZ position immediately before the
    first frame in `root_features`.

    Supported shapes:
    - Single frame: `(..., 3)` with `prev_root_pos` shape `(..., 3)`
    - Sequence: `(..., T, 3)` with `prev_root_pos` shape `(..., 3)`
    """
    if root_features.size(-1) != 3:
        raise ValueError(
            "root_features must end with 3 values: [root_height_y, root_vel_x, root_vel_z]"
        )
    if prev_root_pos.size(-1) != 3:
        raise ValueError("prev_root_pos must end with 3 values: [x, y, z]")

    root_height_y = root_features[..., 0:1]
    root_vel_x = root_features[..., 1:2]
    root_vel_z = root_features[..., 2:3]

    if root_features.ndim == prev_root_pos.ndim:
        root_pos_x = prev_root_pos[..., 0:1] + root_vel_x
        root_pos_z = prev_root_pos[..., 2:3] + root_vel_z
    elif root_features.ndim == prev_root_pos.ndim + 1:
        prev_root_pos = prev_root_pos.unsqueeze(-2)
        root_pos_x = prev_root_pos[..., 0:1] + torch.cumsum(root_vel_x, dim=-2)
        root_pos_z = prev_root_pos[..., 2:3] + torch.cumsum(root_vel_z, dim=-2)
    else:
        raise ValueError(
            "Expected root_features to be either frame-shaped (..., 3) or sequence-shaped (..., T, 3) "
            "relative to prev_root_pos (..., 3)"
        )

    return torch.cat([root_pos_x, root_height_y, root_pos_z], dim=-1)


def root_positions_to_root_features(
    root_positions: torch.Tensor,
    prev_root_pos: torch.Tensor,
) -> torch.Tensor:
    """
    Convert absolute root positions into velocity-form root features.

    The returned layout is `[root_height_y, root_vel_x, root_vel_z]`.
    The first velocity entry is measured against `prev_root_pos`.

    Supported shapes:
    - Single frame: `(..., 3)` with `prev_root_pos` shape `(..., 3)`
    - Sequence: `(..., T, 3)` with `prev_root_pos` shape `(..., 3)`
    """
    if root_positions.size(-1) != 3:
        raise ValueError("root_positions must end with 3 values: [x, y, z]")
    if prev_root_pos.size(-1) != 3:
        raise ValueError("prev_root_pos must end with 3 values: [x, y, z]")

    root_height_y = root_positions[..., 1:2]

    if root_positions.ndim == prev_root_pos.ndim:
        root_vel_x = root_positions[..., 0:1] - prev_root_pos[..., 0:1]
        root_vel_z = root_positions[..., 2:3] - prev_root_pos[..., 2:3]
    elif root_positions.ndim == prev_root_pos.ndim + 1:
        prev_root_pos = prev_root_pos.unsqueeze(-2)
        root_vel_x = torch.cat(
            [
                root_positions[..., :1, 0:1] - prev_root_pos[..., 0:1],
                root_positions[..., 1:, 0:1] - root_positions[..., :-1, 0:1],
            ],
            dim=-2,
        )
        root_vel_z = torch.cat(
            [
                root_positions[..., :1, 2:3] - prev_root_pos[..., 2:3],
                root_positions[..., 1:, 2:3] - root_positions[..., :-1, 2:3],
            ],
            dim=-2,
        )
    else:
        raise ValueError(
            "Expected root_positions to be either frame-shaped (..., 3) or sequence-shaped (..., T, 3) "
            "relative to prev_root_pos (..., 3)"
        )

    return torch.cat([root_height_y, root_vel_x, root_vel_z], dim=-1)


def extract_rotation_target(frame: torch.Tensor) -> torch.Tensor:
    """271D normalized frame -> 135D rotation target."""
    return torch.cat([frame[:, 0:3], frame[:, 69:201]], dim=-1)


def rotation_to_271d(
    rot_pred: torch.Tensor,  # (B, 135) predicted root features + rot6d
    prev_frame: torch.Tensor,  # (B, 271) normalized frame for FK reference
    prev_root_pos: torch.Tensor,  # (B, 3) absolute root position immediately before prev_frame
    normalizer: Normalizer,
) -> tuple[torch.Tensor, torch.Tensor]:
    """135D normalized rotation prediction -> normalized 271D frame for GRU input.

    Uses FK with fixed bone lengths (calibrated once from the first seed frame)
    so articulation can change over time while preserving skeleton structure.
    """
    B = rot_pred.shape[0]
    device = rot_pred.device
    dtype = rot_pred.dtype

    pred_raw = torch.zeros(B, 271, device=device, dtype=dtype)
    pred_raw[:, 0:3] = rot_pred[:, :3]
    pred_raw[:, 69:201] = rot_pred[:, 3:]
    pred_raw = normalizer.denormalize(pred_raw)

    root_features = pred_raw[:, :3]

    root_pos = root_features_to_root_positions(root_features, prev_root_pos)
    rot6d = pred_raw[:, 69:201].reshape(B, 22, 6)

    # Build parent list once from kinematic chains
    if not hasattr(rotation_to_271d, "_parents"):
        parents = [-1] * 22
        for chain in T2M_KINEMATIC_CHAIN:
            for i in range(1, len(chain)):
                parents[chain[i]] = chain[i - 1]
        rotation_to_271d._parents = parents

    # Calibrate fixed bone lengths from the first available frame
    if (
        not hasattr(rotation_to_271d, "_calib_offsets")
        or rotation_to_271d._calib_offsets.device != device
        or rotation_to_271d._calib_offsets.dtype != dtype
    ):
        ref_raw = normalizer.denormalize(prev_frame[:1]).to(device=device, dtype=dtype)
        ref_pos = ref_raw[:, 3:69].reshape(1, 22, 3)[0]
        raw_offsets = T2M_RAW_OFFSETS.to(device=device, dtype=dtype)
        base_dir = raw_offsets / raw_offsets.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        calib_offsets = raw_offsets.clone()
        for j in range(1, 22):
            p = rotation_to_271d._parents[j]
            bone_len = torch.norm(ref_pos[j] - ref_pos[p]).clamp_min(1e-6)
            calib_offsets[j] = base_dir[j] * bone_len
        rotation_to_271d._calib_offsets = calib_offsets

    offsets = rotation_to_271d._calib_offsets
    joints = _forward_kinematics(rot6d, root_pos, offsets, T2M_KINEMATIC_CHAIN)

    root_quat = cont6d_to_quaternion(rot6d[:, 0])
    root_quat_exp = root_quat.unsqueeze(1).expand(-1, 22, -1)
    ric = qrot(root_quat_exp, joints - root_pos.unsqueeze(1))

    prev_raw = normalizer.denormalize(prev_frame).to(device=device, dtype=dtype)
    prev_raw[:, :3] = (
        prev_root_pos  # Override root features with absolute positions for FK reference
    )

    prev_pos = features_to_positions(prev_raw)
    local_vel = qrot(root_quat_exp, joints - prev_pos)

    # Foot contacts from local foot speeds
    feet_l = (torch.sum(local_vel[:, [7, 10]] ** 2, dim=-1) < 0.002).to(dtype)
    feet_r = (torch.sum(local_vel[:, [8, 11]] ** 2, dim=-1) < 0.002).to(dtype)
    contacts = torch.cat([feet_l, feet_r], dim=-1)

    frame_raw = torch.zeros(B, 271, device=device, dtype=dtype)
    frame_raw[:, 0:3] = root_features
    frame_raw[:, 3:69] = ric.reshape(B, 66)
    frame_raw[:, 69:201] = rot6d.reshape(B, 132)
    frame_raw[:, 201:267] = local_vel.reshape(B, 66)
    frame_raw[:, 267:271] = contacts

    return normalizer.normalize(frame_raw), root_pos


def _split_prefixed_state_dict(state_dict, prefix):
    if not isinstance(state_dict, dict):
        return None
    out = {}
    p = f"{prefix}."
    for k, v in state_dict.items():
        if isinstance(k, str) and k.startswith(p):
            out[k[len(p) :]] = v
    return out if out else None


def _pick_state_dict(ckpt, key_candidates, prefix_candidates):
    for key in key_candidates:
        if key in ckpt and isinstance(ckpt[key], dict):
            return ckpt[key], f"ckpt['{key}']"

    for key in ["state_dict", "model", "model_state_dict"]:
        flat_sd = ckpt.get(key)
        if isinstance(flat_sd, dict):
            for prefix in prefix_candidates:
                prefixed = _split_prefixed_state_dict(flat_sd, prefix)
                if prefixed is not None:
                    return prefixed, f"ckpt['{key}'] with prefix '{prefix}.'"

    return None, None


checkpoint_path = Path(f"./checkpoints/stage3_flow_d{config.motion_dim}.pt")
if not checkpoint_path.exists():
    ckpt_dir = checkpoint_path.parent
    ckpt_candidates = sorted(
        ckpt_dir.glob("*.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if ckpt_candidates:
        checkpoint_path = ckpt_candidates[0]
        print(f"Default checkpoint not found; using latest: {checkpoint_path}")

start_epoch = 0
if checkpoint_path.exists():
    print(f"Loading checkpoint: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device)
    if not isinstance(ckpt, dict):
        raise ValueError(
            "Checkpoint is not a dict. Please export a dict-based PyTorch checkpoint."
        )

    def _load_module_state(module, sd, label):
        if sd is None:
            print(f"{label}: not found in checkpoint.")
            return False
        missing, unexpected = module.load_state_dict(sd, strict=False)
        print(
            f"{label}: loaded with strict=False | "
            f"missing={len(missing)} unexpected={len(unexpected)}"
        )
        return True

    gru_sd, gru_src = _pick_state_dict(
        ckpt,
        key_candidates=["gru", "gru_state_dict", "model_gru"],
        prefix_candidates=["gru", "model.gru"],
    )
    flow_sd, flow_src = _pick_state_dict(
        ckpt,
        key_candidates=["flow", "flow_state_dict", "model_flow", "flownet"],
        prefix_candidates=["flow", "model.flow", "flownet", "model.flownet"],
    )

    gru_loaded = _load_module_state(gru, gru_sd, "GRU")
    flow_loaded = _load_module_state(flow, flow_sd, "FlowNet")
    if gru_loaded and gru_src is not None:
        print(f"  GRU source: {gru_src}")
    if flow_loaded and flow_src is not None:
        print(f"  Flow source: {flow_src}")

    gru_ema_sd, _ = _pick_state_dict(
        ckpt,
        key_candidates=["gru_ema", "gru_ema_state_dict", "ema_gru"],
        prefix_candidates=["gru_ema", "ema.gru"],
    )
    flow_ema_sd, _ = _pick_state_dict(
        ckpt,
        key_candidates=["flow_ema", "flow_ema_state_dict", "ema_flow", "ema_flownet"],
        prefix_candidates=["flow_ema", "ema.flow", "ema.flownet"],
    )

    if not _load_module_state(gru_ema.model, gru_ema_sd, "GRU EMA"):
        gru_ema.model.load_state_dict(gru.state_dict(), strict=False)
        print("GRU EMA: initialized from current GRU weights.")
    if not _load_module_state(flow_ema.model, flow_ema_sd, "Flow EMA"):
        flow_ema.model.load_state_dict(flow.state_dict(), strict=False)
        print("Flow EMA: initialized from current FlowNet weights.")

    if "optimizer" in ckpt and isinstance(ckpt["optimizer"], dict):
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
            print("Optimizer state restored.")
        except Exception as e:
            print(f"Optimizer state skipped: {e}")

    if device.type == "cuda" and "scaler" in ckpt and ckpt["scaler"] is not None:
        try:
            scaler.load_state_dict(ckpt["scaler"])
            print("GradScaler state restored.")
        except Exception as e:
            print(f"GradScaler state skipped: {e}")

    if "epoch" in ckpt:
        start_epoch = int(ckpt["epoch"]) + 1
    if "current_horizon" in ckpt:
        current_horizon = int(ckpt["current_horizon"])

    print(
        f"Checkpoint restore summary | start_epoch={start_epoch} | "
        f"current_horizon={current_horizon}"
    )
else:
    print(f"No checkpoint found at {checkpoint_path}. Models use initialized weights.")


def compute_loss(batch, ss_prob=0.0):
    motion = batch["motion"].to(device)
    joints = batch["joints"].to(device)
    lengths = batch["lengths"].to(device)
    captions = batch["captions"]
    B = motion.shape[0]

    with torch.no_grad():
        text_emb_cond = encode_text(captions)

    null_emb = torch.zeros_like(text_emb_cond)
    keep = torch.rand(B, 1, device=device) > cfg_dropout
    text_emb = torch.where(keep, text_emb_cond, null_emb)

    min_length = int(lengths.min().item())
    effective_horizon = min(current_horizon, min_length - 1)
    if effective_horizon <= 0:
        return torch.zeros((), device=device, dtype=motion.dtype)
    max_start = max(1, min_length - effective_horizon - 1)
    start_idx = torch.randint(0, max_start, (1,)).item()

    h = gru.init_hidden(text_emb)
    x_t = motion[:, start_idx]

    if start_idx == 0:
        prev_root_pos = torch.zeros(B, 3, device=device)
    else:
        prev_root_pos = motion[:, start_idx - 1, :3]

    total_loss = torch.tensor(0.0, device=device)

    for step_idx in range(effective_horizon):
        target_frame = motion[:, start_idx + step_idx + 1]
        mask_bool = lengths.reshape(-1) > start_idx + step_idx + 1
        mask_t = mask_bool.float()

        if mask_t.sum().item() == 0:
            x_t = target_frame
            prev_root_pos = joints[:, start_idx + step_idx, :3]
            continue

        h_t, h = gru.step(x_t, text_emb, h)

        clean_target = extract_rotation_target(target_frame)[mask_bool]
        t_flow = torch.rand(clean_target.shape[0], device=device)
        t_flow_ = t_flow.view(clean_target.shape[0], 1)
        noise = torch.randn_like(clean_target)
        x_noisy = (1 - t_flow_) * noise + t_flow_ * clean_target
        v_pred = flow(h_t[mask_bool], x_noisy, t_flow)
        v_target = clean_target - noise
        mse = torch.nn.functional.mse_loss(v_pred, v_target)
        total_loss = total_loss + mse

        if ss_prob > 0.0:
            with torch.no_grad():
                x_roll = torch.randn(B, 135, device=device)
                dt = 1.0 / ss_rollout_steps
                for i in range(ss_rollout_steps):
                    t_roll = torch.full((B,), i * dt, device=device)
                    v_roll = flow(h_t, x_roll, t_roll)
                    x_roll = x_roll + v_roll * dt
                next_norm, _ = rotation_to_271d(
                    x_roll, target_frame, prev_root_pos, normalizer
                )
            use_pred = torch.rand(B, 1, device=device) < ss_prob
            x_t = torch.where(use_pred.expand(-1, 271), next_norm, target_frame)
            x_t_raw = normalizer.denormalize(x_t)
            prev_root_pos = root_features_to_root_positions(
                x_t_raw[:, :3], prev_root_pos
            )
        else:
            x_t = target_frame
            prev_root_pos = joints[:, start_idx + step_idx, :3]

    return total_loss / effective_horizon


def train_epoch(loader, ss_prob):
    gru.train()
    flow.train()
    total_loss = 0.0
    num_batches = 0
    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            loss = compute_loss(batch, ss_prob=ss_prob)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            list(gru.parameters()) + list(flow.parameters()), 1.0
        )
        scaler.step(optimizer)
        scaler.update()
        gru_ema.update(gru)
        flow_ema.update(flow)
        total_loss += loss.item()
        num_batches += 1
    return total_loss / max(1, num_batches)


def validate(loader):
    gru.eval()
    flow.eval()
    total_loss = 0.0
    num_batches = 0
    with torch.no_grad():
        for batch in loader:
            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                loss = compute_loss(batch, ss_prob=0.0)
            total_loss += loss.item()
            num_batches += 1
    return total_loss / max(1, num_batches)


if run_training:
    for epoch in range(start_epoch, num_epochs):
        if epoch > 0 and epoch % curriculum_step_epochs == 0:
            prev = current_horizon
            current_horizon = min(current_horizon + curriculum_step, max_horizon)
            if current_horizon != prev:
                print(f"Curriculum: horizon {prev} -> {current_horizon}")

        if epoch < ss_start_epoch:
            ss_prob = 0.0
        else:
            progress = min(1.0, (epoch - ss_start_epoch) / ss_warmup_epochs)
            ss_prob = ss_max_prob * progress

        train_loss = train_epoch(train_loader, ss_prob)
        val_loss = validate(val_loader)

        torch.save(
            {
                "epoch": epoch,
                "gru": gru.state_dict(),
                "flow": flow.state_dict(),
                "gru_ema": gru_ema.model.state_dict(),
                "flow_ema": flow_ema.model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict() if device.type == "cuda" else None,
                "current_horizon": current_horizon,
            },
            checkpoint_path,
        )

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"horizon={current_horizon} | ss={ss_prob:.3f} | "
            f"Train: {train_loss:.6f} | Val: {val_loss:.6f}"
        )
else:
    print("run_training=False: skipped training loop. Ready for inference cells.")

In [ ]:
def generate(
    gru,
    flow,
    text_emb,
    normalizer=None,
    num_frames=100,
    num_steps=20,
    guidance_scale=3.0,
    seed_frame=None,
    device="cuda",
    noise_seed=None,
):
    gru.eval()
    flow.eval()

    with torch.no_grad():
        if normalizer is None:
            normalizer = globals().get("normalizer", None)
        if normalizer is None:
            raise ValueError("A Normalizer instance is required for generate().")

        text_emb = torch.as_tensor(text_emb, device=device, dtype=torch.float32)
        B = text_emb.shape[0]
        text_emb_p = text_emb.squeeze(1) if text_emb.ndim == 3 else text_emb
        null_emb = torch.zeros_like(text_emb_p)

        h_cond = gru.init_hidden(text_emb_p)
        h_uncond = gru.init_hidden(null_emb)

        if seed_frame is not None:
            x_t = torch.as_tensor(seed_frame, device=device, dtype=torch.float32)
            if x_t.ndim == 1:
                x_t = x_t.unsqueeze(0).expand(B, -1)
        else:
            x_t = torch.zeros(B, 271, device=device)

        rng = None
        if noise_seed is not None:
            rng = torch.Generator(device=device)
            rng.manual_seed(int(noise_seed))

        generated = []

        prev_root_pos = torch.zeros_like(x_t[..., :3])

        for _ in range(num_frames):
            h_t_cond, h_cond = gru.step(x_t, text_emb_p, h_cond)
            h_t_uncond, h_uncond = gru.step(x_t, null_emb, h_uncond)

            x = torch.randn(B, 135, device=device, generator=rng)
            dt = 1.0 / num_steps
            for i in range(num_steps):
                t = torch.full((B,), i * dt, device=device)
                v_cond = flow(h_t_cond, x, t)
                v_uncond = flow(h_t_uncond, x, t)
                x = x + (v_uncond + guidance_scale * (v_cond - v_uncond)) * dt

            x_t_norm, root_pos = rotation_to_271d(x, x_t, prev_root_pos, normalizer)
            x_t_raw = normalizer.denormalize(x_t_norm)
            x_t_raw[:, :3] = root_pos
            joints = features_to_positions(x_t_raw)
            generated.append(joints)
            prev_root_pos = root_pos
            x_t = x_t_norm

        return torch.stack(generated, dim=1)

In [ ]:
# Example usage (after training or checkpoint load)
# captions = ["a person is walking forward"]
# text_emb = encode_text(captions)
# joints = generate(gru_ema.model, flow_ema.model, text_emb, num_frames=120, device=device)
# print(joints.shape)  # (B, T, 22, 3)

# Visualization

This cell provides a concise caption-conditioned rollout visualization using the current inference models.

In [ ]:
# Select inference models (prefer EMA if available)
gru_infer = gru_ema.model if "gru_ema" in globals() else gru
flow_infer = flow_ema.model if "flow_ema" in globals() else flow

# Visualize caption-conditioned rollouts
sample = val_dataset[0]
seed_motion = sample[2]  # motion (T, C) normalized
if torch.is_tensor(seed_motion) and seed_motion.dim() == 3:
    seed_motion = seed_motion[0]
seed_frame = seed_motion[0]
gt_joints = sample[3]
if torch.is_tensor(gt_joints):
    gt_joints = gt_joints.cpu()
if gt_joints.ndim == 4:
    gt_joints = gt_joints[0]
gt_joints = gt_joints.numpy()

captions = [
    "a person is walking forward",
    "a person is waving both arms",
]

# NOW USE SEEDED GENERATION (from ground truth) to show proper motion
# The model can now generate valid poses because RIC comes from previous frame
use_seed = True
seed_for_gen = seed_frame if use_seed else None

rollout_steps = 180
shared_noise_seed = 12345

for caption in captions:
    text_emb = encode_text([caption])
    generated_joints = generate(
        gru_infer,
        flow_infer,
        text_emb,
        normalizer,
        num_frames=rollout_steps,
        num_steps=40,
        guidance_scale=2.5,
        seed_frame=seed_for_gen,
        device=device,
        noise_seed=shared_noise_seed,
    )[0]

    if torch.is_tensor(generated_joints):
        generated_joints = generated_joints.detach().cpu().numpy()

    vel = np.linalg.norm(generated_joints[1:] - generated_joints[:-1], axis=-1)
    root_travel = np.linalg.norm(generated_joints[-1, 0] - generated_joints[0, 0])
    mode = "seeded" if use_seed else "unseeded"
    print(
        f"Caption: {caption} ({mode}) | mean vel: {vel.mean():.6f} | max vel: {vel.max():.6f} | root travel: {root_travel:.6f}"
    )

    # Display-space transform for visualizer convention (no axis swap now - data should be correct)
    generated_vis = generated_joints.copy()
    gt_vis = gt_joints[: generated_joints.shape[0]]

    ani_world = visualize_motion(
        generated_vis,
        ground_truth=gt_vis,
        title=f"World View ({mode}): {caption}",
        notebook=True,
        fps=24,
        skip_frames=1,
    )
    display(ani_world)

    generated_centered = generated_vis - generated_vis[:, 0:1, :]
    gt_centered = gt_vis - gt_vis[:, 0:1, :]
    ani_centered = visualize_motion(
        generated_centered,
        ground_truth=gt_centered,
        title=f"Root-Centered ({mode}): {caption}",
        notebook=True,
        fps=24,
        skip_frames=1,
    )
    display(ani_centered)